# Candidate composition campaign

Parallel and diffusion ASR decoders produce K candidate transcripts per utterance at little extra
cost. This notebook measures, at scale, whether composing them word by word (ROVER over a
confusion network) beats picking one of them whole, across:

- **models**: Whisfusion (masked diffusion), Drax (discrete flow matching); controls: Whisper-small
  and Whisper-large-v3-turbo (autoregressive: greedy, beam n-best, samples) and Parakeet-CTC-1.1B
  (one-step parallel, sampled CTC paths);
- **data**: LibriSpeech dev/test, the Open ASR Leaderboard test sets (AMI, Earnings22, VoxPopuli,
  GigaSpeech, SPGISpeech, Common Voice), FLEURS (en + de/fr/es/it/pt for Drax), SLR83 accents, and
  a controlled babble/white-noise ladder on test-clean.

Two independent worker processes, one per T4, pull jobs from a tiered queue: every cell of the
design gets the same coverage first, then more. Everything is written incrementally; results land
in `/kaggle/working/campaign/results/` (`REPORT.md`, `campaign_summary.json`, per-utterance tables).

**Accelerator: GPU T4 x2. Internet: on.** Nothing needs to be attached.


**Variant: branch-smoke.**

In [ ]:
import time
NB_START = time.time()
MODE = 'full'
RUN_HOURS = 0.9   # whole notebook, setup included; Kaggle stops at 12 h
CONFIG = {'plan': 'tree',
 'tail_minutes': 6,
 'models': ['whisfusion'],
 'tree_sets': ['ls-test-other'],
 'tree_shards': 1,
 'shard_size': 24,
 'tree_mask_mix': 0.5}


Code: every module the campaign runs, written to `/kaggle/working/code/src`.

In [ ]:
import os
for d in ['/kaggle/working/code/src/campaign']:
    os.makedirs(d, exist_ok=True)


In [ ]:
%%writefile /kaggle/working/code/src/scorers.py
"""Training-free ways to pick one of the K candidates. mean_conf is what upstream does."""

from __future__ import annotations

import math
import re

from rapidfuzz.distance import Levenshtein

_PUNCT = re.compile(r"[^\w\s]")


def normalize(text: str) -> str:
    """Must match upstream evaluate_whisfusion.py or our WER is not comparable."""
    return " ".join(_PUNCT.sub("", text.lower()).split())


def edit_distance(a: list, b: list) -> int:
    return Levenshtein.distance(a, b)


def wer_pair(ref: str, hyp: str) -> float:
    r, h = normalize(ref).split(), normalize(hyp).split()
    if not r:
        return 0.0 if not h else 1.0
    return edit_distance(r, h) / len(r)


def corpus_wer(refs: list[str], hyps: list[str]) -> float:
    """Total edits over total reference words."""
    e = w = 0
    for ref, hyp in zip(refs, hyps):
        r = normalize(ref).split()
        e += edit_distance(r, normalize(hyp).split())
        w += len(r)
    return 100.0 * e / max(w, 1)


def mean_utt_wer(refs: list[str], hyps: list[str]) -> float:
    """Mean per-utterance WER; upstream reports this, so compare 8.3% against it."""
    if not refs:
        return 0.0
    return 100.0 * sum(wer_pair(r, h) for r, h in zip(refs, hyps)) / len(refs)


# Scorers take the candidate dicts written by dump_candidates.py and return a
# score per candidate; higher is better.

def s_mean_conf(cands):
    return [c["avg_conf"] for c in cands]


def s_min_conf(cands):
    return [c["min_conf"] for c in cands]


def s_median_conf(cands):
    return [c["median_conf"] for c in cands]


def s_mean_logprob(cands):
    return [c["mean_logprob"] for c in cands]


def s_neg_entropy(cands):
    return [-c["mean_entropy"] for c in cands]


def s_len_norm_conf(cands):
    return [c["avg_conf"] * math.log(1 + c["n_tokens"]) for c in cands]


def s_mbr_wer(cands):
    """MBR within the candidate set: negative mean WER against the others."""
    toks = [normalize(c["text"]).split() for c in cands]
    n = len(toks)
    out = []
    for i in range(n):
        tot = sum(edit_distance(toks[j], toks[i]) / max(len(toks[j]), 1)
                  for j in range(n) if j != i)
        out.append(-tot / max(n - 1, 1))
    return out


def s_conf_plus_mbr(cands, lam: float = 0.5):
    return [a + lam * b for a, b in zip(s_mean_conf(cands), s_mbr_wer(cands))]


SCORERS = {
    "mean_conf (upstream)": s_mean_conf,
    "min_conf": s_min_conf,
    "median_conf": s_median_conf,
    "mean_logprob": s_mean_logprob,
    "neg_entropy": s_neg_entropy,
    "len_norm_conf": s_len_norm_conf,
    "mbr_wer": s_mbr_wer,
    "conf+0.5*mbr": s_conf_plus_mbr,
}


def diversity(cands) -> dict:
    """Low diversity caps what any selection method can achieve."""
    toks = [normalize(c["text"]).split() for c in cands]
    n = len(toks)
    if n < 2:
        return {"mean_pairwise_wer": 0.0, "n_unique": 1}
    tot = cnt = 0.0
    for i in range(n):
        for j in range(i + 1, n):
            tot += edit_distance(toks[i], toks[j]) / max(len(toks[j]), 1)
            cnt += 1
    return {
        "mean_pairwise_wer": 100.0 * tot / cnt,
        "n_unique": len({" ".join(t) for t in toks}),
    }


In [ ]:
%%writefile /kaggle/working/code/src/compose.py
"""Combining candidates word by word instead of picking one of them whole.

Candidates are aligned into a confusion network against the most central one; each slot
then holds the words the candidates propose at that position, plus an epsilon for the ones
that skip it. Voting over the slots is ROVER (Fiscus 1997); the oracle path through the
network is the ceiling that voting could reach.
"""

from __future__ import annotations

import random
from collections import defaultdict

from rapidfuzz.distance import Levenshtein

from scorers import normalize

EPS = ""


def central_index(cands: list[list[str]]) -> int:
    """Candidate with the smallest total distance to the others."""
    best, best_i = None, 0
    for i, a in enumerate(cands):
        tot = sum(Levenshtein.distance(b, a) / max(len(b), 1)
                  for j, b in enumerate(cands) if j != i)
        if best is None or tot < best:
            best, best_i = tot, i
    return best_i


def cluster_weights(cands: list[list[str]], gamma: float = 1.0,
                    threshold: float = 0.0) -> list[float]:
    """Vote weight per candidate, so a cluster of m near-identical candidates counts m**gamma.

    gamma=1 is one vote each, which is what ROVER does and assumes the candidates are
    independent. They are not: the decoder repeats itself. gamma=0 collapses each cluster
    to a single vote.
    """
    n = len(cands)
    parent = list(range(n))

    def find(i):
        while parent[i] != i:
            parent[i] = parent[parent[i]]
            i = parent[i]
        return i

    for i in range(n):
        for j in range(i + 1, n):
            if cands[i] == cands[j]:
                close = True
            elif threshold > 0:
                span = max(len(cands[i]), len(cands[j]), 1)
                close = Levenshtein.distance(cands[i], cands[j]) / span <= threshold
            else:
                close = False
            if close:
                a, b = find(i), find(j)
                if a != b:
                    parent[b] = a

    size: dict[int, int] = defaultdict(int)
    for i in range(n):
        size[find(i)] += 1
    return [float(size[find(i)]) ** (gamma - 1.0) for i in range(n)]


def confusion_network(cands: list[list[str]], backbone: int,
                      confs: list[list[float]] | None = None,
                      weights: list[float] | None = None) -> list[dict]:
    """Align every candidate to the backbone. Returns one dict per slot mapping
    word -> [summed vote weight, summed weighted confidence]."""
    bb = cands[backbone]
    n = len(bb)
    w = weights if weights is not None else [1.0] * len(cands)
    k = sum(w)
    sub: list[dict] = [defaultdict(lambda: [0.0, 0.0]) for _ in range(n)]
    ins: list[dict] = [defaultdict(lambda: [0.0, 0.0]) for _ in range(n + 1)]
    ins_voters: list[set] = [set() for _ in range(n + 1)]

    def add(slot, word, cand_i, pos):
        e = slot[word]
        e[0] += w[cand_i]
        if confs is not None and word != EPS and pos is not None:
            c = confs[cand_i]
            e[1] += (c[pos] if pos < len(c) else 1.0) * w[cand_i]
        else:
            e[1] += w[cand_i]

    for ci, c in enumerate(cands):
        for tag, i1, i2, j1, j2 in Levenshtein.opcodes(bb, c):
            if tag == "equal":
                for d in range(i2 - i1):
                    add(sub[i1 + d], c[j1 + d], ci, j1 + d)
            elif tag == "replace":
                for pos in range(i1, i2):
                    off = j1 + (pos - i1)
                    if off < j2:
                        add(sub[pos], c[off], ci, off)
                    else:
                        add(sub[pos], EPS, ci, None)
                for off in range(j1 + (i2 - i1), j2):
                    add(ins[i2], c[off], ci, off)
                    ins_voters[i2].add(ci)
            elif tag == "delete":
                for pos in range(i1, i2):
                    add(sub[pos], EPS, ci, None)
            elif tag == "insert":
                for off in range(j1, j2):
                    add(ins[i1], c[off], ci, off)
                    ins_voters[i1].add(ci)

    # Candidates that inserted nothing at a point still vote there, for epsilon.
    for i in range(n + 1):
        if ins[i]:
            ins[i][EPS] = [k - sum(w[c] for c in ins_voters[i]), 0.0]

    slots = []
    for i in range(n):
        if ins[i]:
            slots.append(dict(ins[i]))
        slots.append(dict(sub[i]))
    if ins[n]:
        slots.append(dict(ins[n]))

    for s in slots:
        s.setdefault(EPS, [0.0, 0.0])
    return slots


def oracle_path(slots: list[dict], ref: list[str]) -> int:
    """Fewest edits any path through the network can achieve against ref."""
    INF = 10 ** 9
    R = len(ref)
    cur = list(range(R + 1))
    for slot in slots:
        nxt = [INF] * (R + 1)
        for j in range(R + 1):
            if cur[j] == INF:
                continue
            for w in slot:
                if w == EPS:
                    nxt[j] = min(nxt[j], cur[j])
                else:
                    nxt[j] = min(nxt[j], cur[j] + 1)          # spurious word
                    if j < R:
                        nxt[j + 1] = min(nxt[j + 1], cur[j] + (0 if w == ref[j] else 1))
        for j in range(R):                                    # skipped reference word
            nxt[j + 1] = min(nxt[j + 1], nxt[j] + 1)
        cur = nxt
    return cur[R]


def rover_slot_picks(slots: list[dict], total: float, alpha: float = 1.0,
                     eps_conf: float = 0.5) -> list[str]:
    """The winning word of every slot, epsilons included, one per slot."""
    out = []
    for slot in slots:
        best, best_w = None, EPS
        for w, (votes, conf_sum) in slot.items():
            freq = votes / total if total else 0.0
            conf = eps_conf if w == EPS else (conf_sum / votes if votes > 0 else 0.0)
            score = alpha * freq + (1.0 - alpha) * conf
            if best is None or score > best:
                best, best_w = score, w
        out.append(best_w)
    return out


def rover(slots: list[dict], total: float, alpha: float = 1.0,
          eps_conf: float = 0.5) -> list[str]:
    """Vote per slot. alpha=1 is pure frequency; below that, confidence weighs in.

    total is the summed vote weight of all candidates, so frequencies stay in [0, 1].
    """
    return [w for w in rover_slot_picks(slots, total, alpha, eps_conf) if w != EPS]


def shuffled_control(slots: list[dict], pool: list[str], rng: random.Random) -> list[dict]:
    """Same network shape, alternatives replaced by unrelated words.

    Isolates how much of the oracle's gain is free choice per slot rather than the
    candidates actually holding the right word.
    """
    out = []
    for slot in slots:
        words = [w for w in slot if w != EPS]
        if not words:
            out.append(dict(slot))
            continue
        keep = words[0]
        new = {keep: list(slot[keep]), EPS: list(slot[EPS])}
        for w in words[1:]:
            new[rng.choice(pool)] = list(slot[w])
        out.append(new)
    return out


def prepare(row: dict, norm=normalize) -> tuple[list[list[str]], list[list[float]] | None]:
    """Candidate word lists, and per-word confidences when the dump carries them.

    norm is the text normaliser the words are compared in, and it must be the same one the
    reference is scored with. Compose in the space you measure in: normalising ROVER's output
    a second time further downstream is not the same as normalising the candidates once, since
    a normaliser that expands contractions cannot expand what an earlier pass already stripped.
    """
    cands = [norm(c["text"]).split() for c in row["candidates"]]
    confs = None
    if all("word_conf" in c for c in row["candidates"]):
        confs = [c["word_conf"] for c in row["candidates"]]
        if any(len(w) != len(t) for w, t in zip(confs, cands)):
            confs = None
    return cands, confs


In [ ]:
%%writefile /kaggle/working/code/src/bootstrap.py
"""Paired bootstrap over utterances, for comparing two systems measured on the same data."""

from __future__ import annotations

import numpy as np


def paired_bootstrap(edits_a: list[int], edits_b: list[int], ref_len: list[int],
                     n_boot: int = 4000, seed: int = 0) -> dict:
    """Confidence interval for corpus WER(a) - WER(b). Positive means b is better.

    Both systems are scored on the same resampled utterances, so the interval covers the
    difference between them and not the spread of the test set, which is much larger.
    """
    a = np.asarray(edits_a, dtype=np.float64)
    b = np.asarray(edits_b, dtype=np.float64)
    r = np.asarray(ref_len, dtype=np.float64)
    n = len(r)
    if n == 0 or r.sum() == 0:
        return {}

    observed = 100.0 * (a.sum() - b.sum()) / r.sum()

    rng = np.random.default_rng(seed)
    deltas = np.empty(n_boot, dtype=np.float64)
    step = 500
    for start in range(0, n_boot, step):
        idx = rng.integers(0, n, size=(min(step, n_boot - start), n))
        sr = r[idx].sum(axis=1)
        deltas[start:start + idx.shape[0]] = 100.0 * (a[idx].sum(axis=1) - b[idx].sum(axis=1)) / sr

    return {
        "delta": observed,
        "ci_low": float(np.percentile(deltas, 2.5)),
        "ci_high": float(np.percentile(deltas, 97.5)),
        "p_no_gain": float((deltas <= 0).mean()),
        "n_better": int((b < a).sum()),
        "n_worse": int((b > a).sum()),
        "n_tied": int((a == b).sum()),
        "n_boot": n_boot,
    }


def format_row(label: str, s: dict) -> str:
    if not s:
        return f"{label:<34} (empty)"
    return (f"{label:<34}{s['delta']:+7.2f}  95% CI [{s['ci_low']:+.2f}, {s['ci_high']:+.2f}]"
            f"   better {s['n_better']} / worse {s['n_worse']} / tied {s['n_tied']}")


In [ ]:
%%writefile /kaggle/working/code/src/decode.py
"""Parallel Diffusion Decoding, returning all K candidates instead of the winner.

Step 0 masks everything and the update is argmax, so all K candidates are identical after
it; divergence comes from the masks of later steps. first_step_sampling samples there.
"""

from __future__ import annotations

from dataclasses import asdict, dataclass, field

import torch

DEFAULT_SCHEDULE = [1.0, 0.9, 0.85, 0.8]
WORD_START = "\u2581"  # SentencePiece word boundary


def _word_confidences(tokenizer, ids, confs, text) -> list[float] | None:
    """Mean confidence per normalised word, or None if it will not line up."""
    from scorers import normalize

    special = set(tokenizer.all_special_ids)
    pairs = [(i, c) for i, c in zip(ids, confs) if i not in special]
    if not pairs:
        return None
    keep_ids, keep_conf = zip(*pairs)
    pieces = tokenizer.convert_ids_to_tokens(list(keep_ids))

    words, buf, cur = [], [], []
    for piece, c in zip(pieces, keep_conf):
        if piece.startswith(WORD_START) and buf:
            words.append(("".join(buf), cur))
            buf, cur = [], []
        buf.append(piece.lstrip(WORD_START))
        cur.append(float(c))
    if buf:
        words.append(("".join(buf), cur))

    out = [sum(cs) / len(cs) for w, cs in words if normalize(w)]
    return out if len(out) == len(normalize(text).split()) else None


@dataclass
class Candidate:
    text: str
    avg_conf: float  # upstream selection metric: mean max-prob over non-pad positions
    min_conf: float
    median_conf: float
    mean_logprob: float
    mean_entropy: float
    n_tokens: int
    word_conf: list[float] | None = field(default=None)
    tokens: list[int] | None = field(default=None)


@dataclass
class DecodeResult:
    candidates: list[Candidate]
    n_unique: int
    identical_after_step1: bool
    n_candidates_used: int = 0      # differs from the request only when adaptive is on
    uncertainty: float | None = None  # mean (1 - max prob) at the probe step, if measured
    branch_widths: list[int] | None = None   # rows actually decoded at each step


@torch.no_grad()
def pdd_decode(
    wf,
    condition: torch.Tensor,
    n_candidates: int = 15,
    n_steps: int = 4,
    mask_ratio_schedule: list[float] | None = None,
    seq_len: int = 256,
    first_step_sampling: bool = False,
    temperature: float = 1.0,
    save_tokens: bool = False,
    seed: int | None = None,
    branch_schedule: list[int] | None = None,
    mask_mode: str = "uniform",
    mask_mix: float = 1.0,
    adaptive: dict | None = None,
) -> DecodeResult:
    """branch_schedule gives the number of distinct mask groups at each step.

    Flat sampling is every candidate drawing its own mask at every step, which is
    branch_schedule = [K, K, K, K] and is what None means. A tree shares masks early and
    splits later: [1, 4, 32, 32] decodes one row, then four, then thirty-two, so siblings
    carry a common prefix. Rows are only materialised when they split, so a narrow early
    schedule is also cheaper, not just more correlated.

    mask_mode "uncertain" draws the mask with probability proportional to 1 - confidence
    from the previous step instead of uniformly, keeping the same expected mask ratio, so
    re-prediction is spent where the model is unsure rather than spread evenly.

    adaptive sizes the whole tree per utterance: after probe_step the width is set from how
    uncertain the shared prefix is, so an easy utterance gets a small tree and a hard one a
    large tree. dict(base, u0, gamma, k_min, k_max, probe_step); K = base * (u/u0) ** gamma.
    """
    schedule = mask_ratio_schedule or DEFAULT_SCHEDULE
    device = wf.device
    mask_id = wf.mask_token_id
    pad_id = wf.pad_token_id

    gen = None
    if seed is not None:
        gen = torch.Generator(device=device)
        gen.manual_seed(seed)

    bos = wf.tokenizer.bos_token_id
    bos = 0 if bos is None else bos

    widths = list(branch_schedule) if branch_schedule else [n_candidates] * n_steps
    widths = [max(1, int(w)) for w in widths[:n_steps]]
    widths += [widths[-1]] * (n_steps - len(widths))
    probe = int((adaptive or {}).get("probe_step", 1))

    rows = widths[0]
    cur = torch.full((rows, seq_len), mask_id, dtype=torch.long, device=device)
    cur[:, 0] = bos

    final_logits = None
    after_step1 = None
    conf_prev = None
    score_prev = None
    uncertainty = None
    used = []

    for step in range(n_steps):
        ratio = schedule[step] if step < len(schedule) else 0.7

        want = widths[step]
        if want > rows:                       # split: every parent takes the same children
            idx = torch.arange(want, device=device) % rows if want % rows else None
            cur = cur[idx] if idx is not None else cur.repeat_interleave(want // rows, dim=0)
            if conf_prev is not None:
                conf_prev = conf_prev[idx] if idx is not None else \
                    conf_prev.repeat_interleave(want // rows, dim=0)
        elif want < rows:                     # prune, which is how adaptive width shrinks
            cur = cur[:want]                  # rows are interchangeable at the probe step
            if conf_prev is not None:
                conf_prev = conf_prev[:want]
        rows = want
        used.append(rows)
        cond = condition.expand(rows, -1, -1)

        if ratio <= 0:
            mask_idx = torch.zeros((rows, seq_len), dtype=torch.bool, device=device)
        elif mask_mode != "uniform" and score_prev is not None:
            # Mask exactly as many positions as the uniform draw would, but choose them by
            # weighted sampling WITHOUT replacement. The first version scaled a per-position
            # probability, which at a high mask ratio pinned the low-confidence positions to
            # p = 1: they were re-masked every step, never accumulated context and never
            # settled, and WER came out four times worse. Sampling a fixed count keeps the
            # budget identical to flat and still leaves every position a chance to be spared.
            # Pure targeting starves the confident positions of any revision, so early errors
            # lock in and every candidate converges on the same wrong transcript: that is what
            # cost 30 WER points. mask_mix keeps part of the budget uniform, so exploration
            # never stops; mix = 0 is flat sampling and mix = 1 is the version that failed.
            n_mask = min(int(round(ratio * (seq_len - 1))), seq_len - 1)
            n_aim = int(round(mask_mix * n_mask))
            mask_idx = torch.zeros((rows, seq_len), dtype=torch.bool, device=device)
            if n_aim > 0:
                w = torch.cat([torch.zeros_like(score_prev[:, :1]),
                               score_prev[:, 1:].clamp_min(1e-6)], dim=1)
                mask_idx.scatter_(1, torch.multinomial(w, n_aim, replacement=False,
                                                       generator=gen), True)
            if n_mask > n_aim:
                free = (~mask_idx).float()
                free[:, 0] = 0.0
                mask_idx.scatter_(1, torch.multinomial(free, n_mask - n_aim,
                                                       replacement=False, generator=gen), True)
        else:
            r = torch.rand((rows, seq_len), device=device, generator=gen)
            mask_idx = r < ratio
            mask_idx[:, 0] = False

        masked = cur.clone()
        masked[mask_idx] = mask_id

        logits = wf.model(idx=masked, condition=cond)

        if step == 0 and first_step_sampling:
            probs = torch.softmax(logits.float() / temperature, dim=-1)
            pred = torch.multinomial(probs.view(-1, probs.size(-1)), 1,
                                     generator=gen).view(rows, seq_len)
        else:
            pred = torch.argmax(logits, dim=-1)

        prev_tokens = cur
        cur = torch.where(mask_idx, pred, masked)
        if mask_mode != "uniform" or adaptive is not None:
            probs_p = torch.softmax(logits.float(), dim=-1)
            top2 = probs_p.topk(2, dim=-1).values
            conf_prev = top2[..., 0]
            if mask_mode == "uncertain":
                score_prev = 1.0 - conf_prev
            elif mask_mode == "entropy":
                score_prev = -(probs_p * torch.log(probs_p + 1e-9)).sum(-1)
            elif mask_mode == "margin":                       # a close runner-up is a real doubt
                score_prev = 1.0 - (top2[..., 0] - top2[..., 1])
            elif mask_mode == "unstable":                     # did this token just change
                score_prev = (cur != prev_tokens).float() + 0.05
            elif mask_mode == "disagree":                     # where the siblings already differ
                score_prev = (cur != cur[0:1]).float().mean(0, keepdim=True) \
                    .expand(rows, -1).contiguous() + 0.05
            del probs_p, top2

        if adaptive is not None and step == probe and uncertainty is None:
            a = adaptive
            uncertainty = float((1.0 - conf_prev).mean())
            # base is the TARGET MEAN width, not a ceiling, and u0 is the measured median
            # uncertainty (0.021 over three sets), not a guess. The first version used
            # u0 = 0.15, about seven times too high, so every utterance clipped to k_min.
            k = a.get("base", n_candidates) * (uncertainty / a.get("u0", 0.021)) ** a.get("gamma", 1.0)
            k = int(min(max(round(k), a.get("k_min", 2)), a.get("k_max", n_candidates)))
            scale = k / max(n_candidates, 1)
            for s in range(step + 1, n_steps):       # re-aim the rest of the tree at k
                widths[s] = max(1, min(k, round(widths[s] * scale)))
            widths[n_steps - 1] = k

        if step == 0:
            after_step1 = bool((cur == cur[0:1]).all().item())
        if step == n_steps - 1:
            final_logits = logits

    n_candidates = cur.size(0)

    probs = torch.softmax(final_logits.float(), dim=-1)
    conf = probs.max(dim=-1).values
    logprob = torch.log(
        probs.gather(-1, cur.clamp(max=probs.size(-1) - 1).unsqueeze(-1)).squeeze(-1) + 1e-9
    )
    entropy = -(probs * torch.log(probs + 1e-9)).sum(-1)
    del probs, final_logits

    toks = cur.cpu()
    conf, logprob, entropy = conf.cpu(), logprob.cpu(), entropy.cpu()
    texts = wf.tokenizer.batch_decode(toks, skip_special_tokens=True)
    valid = toks != pad_id

    cands = []
    for i in range(n_candidates):
        v = valid[i]
        if int(v.sum()) == 0:
            v = torch.ones_like(v)
        c = conf[i][v]
        ids = toks[i][v].tolist()
        cands.append(
            Candidate(
                text=texts[i],
                avg_conf=float(c.mean()),
                min_conf=float(c.min()),
                median_conf=float(c.median()),
                mean_logprob=float(logprob[i][v].mean()),
                mean_entropy=float(entropy[i][v].mean()),
                n_tokens=int(v.sum()),
                word_conf=_word_confidences(wf.tokenizer, ids, c.tolist(), texts[i]),
                tokens=ids if save_tokens else None,
            )
        )

    return DecodeResult(
        candidates=cands,
        n_unique=len({c.text for c in cands}),
        identical_after_step1=bool(after_step1),
        n_candidates_used=len(cands),
        uncertainty=uncertainty,
        branch_widths=used,
    )


def candidate_to_dict(c: Candidate) -> dict:
    d = asdict(c)
    for key in ("tokens", "word_conf"):
        if d[key] is None:
            d.pop(key)
    return d


In [ ]:
%%writefile /kaggle/working/code/src/check_decode.py
#!/usr/bin/env python3
"""Flat sampling must be bit-identical to the decoder before branching was added.

The tree parameters default to off, so the old code path has to survive untouched: same
seed, same masks, same text. Everything measured before the tree work was produced by the
old path, so a mismatch means those numbers stopped being comparable.

No dataset is needed. What is under test is the order of the random draws and the update,
not the audio, so a fixed pseudo-random encoder input exercises it exactly as speech would.
The reference decoder is imported as decode_old, which the notebook writes from git.
"""

from __future__ import annotations


def fake_condition(wf, seconds: float = 5.0, seed: int = 0):
    import numpy as np

    import wf_model
    rng = np.random.default_rng(seed)
    audio = (rng.standard_normal(int(16000 * seconds)) * 0.05).astype("float32")
    return wf_model.encode_audio(wf, audio)


def compare(wf, old, new, n_trials: int = 6, k: int = 16, n_steps: int = 4) -> int:
    """Returns the number of mismatching trials; prints one diff for the first."""
    bad = 0
    for t in range(n_trials):
        cond = fake_condition(wf, seconds=3.0 + t, seed=t)
        a = old.pdd_decode(wf, cond, n_candidates=k, n_steps=n_steps, seed=1000 + t)
        b = new.pdd_decode(wf, cond, n_candidates=k, n_steps=n_steps, seed=1000 + t)
        ta = [c.text for c in a.candidates]
        tb = [c.text for c in b.candidates]
        ok = ta == tb
        bad += not ok
        print(f"  trial {t}  {'ok' if ok else 'MISMATCH'}   unique {a.n_unique} -> {b.n_unique}")
        if not ok and bad == 1:
            for i, (x, y) in enumerate(zip(ta, tb)):
                if x != y:
                    print(f"      candidate {i}\n        old: {x[:100]}\n        new: {y[:100]}")
                    break
    print(f"\n{n_trials - bad}/{n_trials} identical"
          + ("  -- flat sampling is unchanged" if not bad else "  -- DO NOT RUN THE ABLATION"))
    return bad


def tree_sanity(wf, new, k: int = 16) -> int:
    """Every arm must decode, branch as told, size itself, and not fall apart.

    The previous run printed all of this and passed anyway, because nothing asserted. Five of
    nine arms were broken: adapt never changed width, and uncertainty masking pinned the hard
    positions to p = 1 so they never settled and WER came out four times worse. Each check
    below is one of those failures turned into a condition.
    """
    cond = fake_condition(wf, seconds=6.0, seed=99)
    q = max(k // 4, 1)
    low = [1.0, 0.5, 0.35, 0.25]
    ad = dict(base=max(k * 2 // 3, 4), u0=0.021, gamma=1.5, k_min=max(k // 6, 2), k_max=k)
    arms = [
        ("flat", dict(), None),
        ("tree-early", dict(branch_schedule=[1, q, k, k]), [1, q, k, k]),
        ("tree-late", dict(branch_schedule=[1, 1, q, k]), [1, 1, q, k]),
        ("flat-sched", dict(mask_ratio_schedule=low), None),
        ("cond", dict(mask_ratio_schedule=low, mask_mode="uncertain"), None),
        ("cond-tree", dict(mask_ratio_schedule=low, mask_mode="uncertain",
                           branch_schedule=[1, q, k, k]), [1, q, k, k]),
        ("adapt", dict(adaptive=ad), None),
        ("adapt-tree", dict(branch_schedule=[1, q, k, k], adaptive=ad), None),
    ]
    print(f"\n{'arm':<13}{'K':>5}{'unique':>8}{'mean len':>10}{'widths':>20}{'uncert.':>10}")
    res, bad = {}, []
    for name, kw, want in arms:
        r = new.pdd_decode(wf, cond, n_candidates=k, n_steps=4, seed=7, **kw)
        res[name] = r
        lens = [c.n_tokens for c in r.candidates]
        mean_len = sum(lens) / max(len(lens), 1)
        u = "" if r.uncertainty is None else f"{r.uncertainty:.4f}"
        print(f"{name:<13}{r.n_candidates_used:>5}{r.n_unique:>8}{mean_len:>10.1f}"
              f"{str(r.branch_widths):>20}{u:>10}")
        if want is not None and r.branch_widths != want:
            bad.append(f"{name}: branched {r.branch_widths}, asked for {want}")

    base_len = sum(c.n_tokens for c in res["flat"].candidates) / k
    for name in ("cond", "cond-tree", "flat-sched"):
        lens = [c.n_tokens for c in res[name].candidates]
        m = sum(lens) / max(len(lens), 1)
        if not 0.5 * base_len <= m <= 2.0 * base_len:   # divergence shows up as length first
            bad.append(f"{name}: mean length {m:.0f} against flat {base_len:.0f}, it is diverging")
    if res["adapt"].n_candidates_used == k:
        bad.append(f"adapt returned the full {k}: the width never adapted (u0 miscalibrated, "
                   f"or rows are not allowed to shrink)")
    if res["adapt"].uncertainty is None:
        bad.append("adapt reported no uncertainty: the probe step never ran")

    for line in bad:
        print(f"  FAIL  {line}")
    print("every arm behaves" if not bad else f"\n{len(bad)} arms misbehaving")
    return len(bad)


def main() -> int:
    import importlib
    import sys

    from campaign.families import Whisfusion
    new = importlib.import_module("decode")
    try:
        old = importlib.import_module("decode_old")
    except ImportError:
        print("decode_old not importable; the notebook writes it from git at build time")
        return 2
    wf = Whisfusion({}).wf
    bad = compare(wf, old, new)
    bad += tree_sanity(wf, new)
    return 1 if bad else 0


if __name__ == "__main__":
    raise SystemExit(main())


In [ ]:
%%writefile /kaggle/working/code/src/wf_model.py
"""Load Whisfusion (frozen Whisper encoder + masked-diffusion decoder) for inference."""

from __future__ import annotations

from dataclasses import dataclass

import torch

import wf_compat

wf_compat.install()

from lit_gpt.diffmodel import TransEncoder, Config  # noqa: E402
from safetensors.torch import load_file  # noqa: E402
from transformers import (  # noqa: E402
    AutoTokenizer,
    WhisperForConditionalGeneration,
    WhisperProcessor,
)

DEFAULT_MODEL_NAME = "Diff_LLaMA_170M"
DEFAULT_TOKENIZER = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"
DEFAULT_WHISPER = "openai/whisper-small"


def pick_dtype(device: str, requested: str = "auto") -> torch.dtype:
    """The paper runs bf16; T4 and P100 do not support it, so we fall back to fp16."""
    if requested != "auto":
        return {"fp16": torch.float16, "bf16": torch.bfloat16, "fp32": torch.float32}[requested]
    if device != "cuda":
        return torch.float32
    # Not is_bf16_supported(): since torch 2.6 it counts emulation, so it says yes on a T4.
    return torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16


@dataclass
class Whisfusion:
    model: TransEncoder
    config: Config
    tokenizer: object
    whisper_processor: object
    whisper_encoder: object
    device: str
    dtype: torch.dtype

    @property
    def mask_token_id(self) -> int:
        return self.config.padded_vocab_size

    @property
    def pad_token_id(self) -> int:
        return self.tokenizer.pad_token_id


def load(
    base_model_path: str,
    adapter_path: str,
    model_name: str = DEFAULT_MODEL_NAME,
    tokenizer_name: str = DEFAULT_TOKENIZER,
    whisper_name: str = DEFAULT_WHISPER,
    device: str | None = None,
    dtype: str = "auto",
    verbose: bool = True,
) -> Whisfusion:
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    torch_dtype = pick_dtype(device, dtype)

    if verbose:
        name = torch.cuda.get_device_name(0) if device == "cuda" else "cpu"
        print(f"[wf] device={device} ({name}) dtype={torch_dtype}")

    config = Config.from_name(model_name)
    model = TransEncoder(config)

    if base_model_path:
        if base_model_path.endswith(".safetensors"):
            base = load_file(base_model_path)
        else:
            base = torch.load(base_model_path, map_location="cpu", weights_only=False)
            base = base.get("state_dict", base)
        res = model.load_state_dict(base, strict=False)
        if verbose:
            print(f"[wf] base: {len(base)} keys, {len(res.missing_keys)} missing")

    # Stage 2 carries the whole decoder including cross-attention, so it overwrites
    # most of the base weights.
    ad = torch.load(adapter_path, map_location="cpu", weights_only=False)
    ad = ad.get("state_dict", ad) if isinstance(ad, dict) else ad
    res = model.load_state_dict(ad, strict=False)
    if verbose:
        print(f"[wf] adapter: {len(ad)} keys, {len(res.missing_keys)} missing, "
              f"{len(res.unexpected_keys)} unexpected")
    if res.missing_keys:
        raise RuntimeError(
            f"uninitialised parameters after loading: {res.missing_keys[:10]}"
            f"{' ...' if len(res.missing_keys) > 10 else ''}"
        )

    model = model.to(device=device, dtype=torch_dtype).eval()

    tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    whisper_processor = WhisperProcessor.from_pretrained(whisper_name)
    whisper = WhisperForConditionalGeneration.from_pretrained(whisper_name)
    whisper = whisper.to(device=device, dtype=torch_dtype).eval()

    if verbose:
        n = sum(p.numel() for p in model.parameters())
        ne = sum(p.numel() for p in whisper.model.encoder.parameters())
        print(f"[wf] decoder {n/1e6:.1f}M · Whisper encoder {ne/1e6:.1f}M")

    return Whisfusion(
        model=model,
        config=config,
        tokenizer=tokenizer,
        whisper_processor=whisper_processor,
        whisper_encoder=whisper.model.encoder,
        device=device,
        dtype=torch_dtype,
    )


@torch.no_grad()
def encode_audio(wf: Whisfusion, audio, sampling_rate: int = 16000) -> torch.Tensor:
    """1-D float audio -> (1, T_cond, n_embd) conditioning for the decoder."""
    inputs = wf.whisper_processor(audio, sampling_rate=sampling_rate, return_tensors="pt")
    feats = inputs.input_features.to(device=wf.device, dtype=wf.dtype)
    return wf.whisper_encoder(feats).last_hidden_state.to(wf.dtype)


In [ ]:
%%writefile /kaggle/working/code/src/wf_compat.py
"""Pure-PyTorch stand-ins for the CUDA extensions lit-gpt/SMDM imports.

FlashAttention 2 needs Ampere; Kaggle serves T4 or P100. Registering replacements in
sys.modules keeps the upstream checkout untouched. Call install() before importing lit_gpt.
"""

from __future__ import annotations

import importlib.machinery
import sys
import types

import torch
import torch.nn as nn
import torch.nn.functional as F

_INSTALLED = False


def _make_module(name: str) -> types.ModuleType:
    # importlib.util.find_spec() raises ValueError on a module whose __spec__ is
    # None, and transformers calls it while probing for flash-attn.
    m = types.ModuleType(name)
    m.__spec__ = importlib.machinery.ModuleSpec(name, None)
    m.__file__ = f"<wf_compat shim: {name}>"
    return m


def _apply_rotary(x1, x2, cos, sin, out1, out2, conj):
    """rotary_emb.apply_rotary. out1 may alias x1, so compute both before writing."""
    a, b = x1.float(), x2.float()
    c, s = cos.float(), sin.float()
    if conj:
        o1, o2 = a * c + b * s, -a * s + b * c
    else:
        o1, o2 = a * c - b * s, a * s + b * c
    out1.copy_(o1.to(out1.dtype))
    out2.copy_(o2.to(out2.dtype))


class SwiGLU(nn.Module):
    """Unpacked xformers.ops.SwiGLU; w1/w2/w3 match the checkpoint keys."""

    def __init__(self, in_features, hidden_features, out_features=None,
                 bias=True, *, _pack_weights=True):
        super().__init__()
        out_features = out_features if out_features is not None else in_features
        self.w1 = nn.Linear(in_features, hidden_features, bias=bias)
        self.w2 = nn.Linear(in_features, hidden_features, bias=bias)
        self.w3 = nn.Linear(hidden_features, out_features, bias=bias)

    def forward(self, x):
        return self.w3(F.silu(self.w1(x)) * self.w2(x))


class RMSNorm(nn.Module):
    """Stand-in for lit_gpt.rmsnorm.FusedRMSNorm."""

    def __init__(self, size: int, dim: int = -1, eps: float = 1e-5):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(size))
        self.eps = eps
        self.dim = dim

    def forward(self, x):
        dtype = x.dtype
        xf = x.float()  # mean(x*x) overflows in fp16 and surfaces as NaN layers later
        xf = xf * torch.rsqrt(torch.mean(xf * xf, dim=self.dim, keepdim=True) + self.eps)
        return xf.to(dtype) * self.weight


def _flash_attn_func(q, k, v, dropout_p=0.0, softmax_scale=None, causal=False, **kw):
    """FlashAttention-2 signature, SDPA underneath. (B, T, H, D) in and out."""
    q, k, v = q.transpose(1, 2), k.transpose(1, 2), v.transpose(1, 2)
    if q.shape[1] != k.shape[1]:  # GQA / MQA
        k = k.repeat_interleave(q.shape[1] // k.shape[1], dim=1)
        v = v.repeat_interleave(q.shape[1] // v.shape[1], dim=1)
    y = F.scaled_dot_product_attention(q, k, v, dropout_p=dropout_p,
                                       scale=softmax_scale, is_causal=causal)
    return y.transpose(1, 2)


def install(verbose: bool = True) -> None:
    global _INSTALLED
    if _INSTALLED:
        return
    if "lit_gpt.diffmodel" in sys.modules or "lit_gpt.model" in sys.modules:
        raise RuntimeError(
            "install() must run before lit_gpt is imported: lit_gpt copies "
            "apply_rotary_emb_func into its own namespace at import time."
        )

    # Let transformers finish its flash-attn probe before the shim shadows it.
    try:
        import transformers  # noqa: F401
    except Exception:
        pass

    if "rotary_emb" not in sys.modules:
        m = _make_module("rotary_emb")
        m.apply_rotary = _apply_rotary
        sys.modules["rotary_emb"] = m

    for name in ("dropout_layer_norm", "xentropy_cuda_lib"):
        # Empty is fine: FusedRMSNorm is replaced below and fused cross-entropy
        # is training-only.
        if name not in sys.modules:
            sys.modules[name] = _make_module(name)

    try:
        import flash_attn  # noqa: F401
    except Exception:
        m = _make_module("flash_attn")
        m.flash_attn_func = _flash_attn_func
        sys.modules["flash_attn"] = m

    try:
        from xformers.ops import SwiGLU as _  # noqa: F401
    except Exception:
        xf = sys.modules.get("xformers") or _make_module("xformers")
        ops = _make_module("xformers.ops")
        ops.SwiGLU = SwiGLU
        xf.ops = ops
        sys.modules["xformers"] = xf
        sys.modules["xformers.ops"] = ops

    # Config.norm_class imports FusedRMSNorm lazily, so patching the attribute is enough.
    import lit_gpt.rmsnorm as _rms

    _rms.FusedRMSNorm = RMSNorm

    _INSTALLED = True
    if verbose:
        print("[wf_compat] shims active: rotary_emb, dropout_layer_norm, "
              "flash_attn, xformers.ops.SwiGLU, FusedRMSNorm")


def assert_no_cuda_ext() -> None:
    """flash-attn installed via pip wins over the shim and dies on T4/P100."""
    from lightning_utilities.core.imports import RequirementCache

    if bool(RequirementCache("flash-attn>=2.0.0.post1")):
        raise RuntimeError("flash-attn is pip-installed; uninstall it: pip uninstall -y flash-attn")


In [ ]:
%%writefile /kaggle/working/code/src/data.py
"""Audio loading via soundfile; `datasets` now needs torchcodec, which breaks on Kaggle."""

from __future__ import annotations

import json
from dataclasses import dataclass
from pathlib import Path
from typing import Iterator

import numpy as np
import soundfile as sf

TARGET_SR = 16000


@dataclass
class Utterance:
    id: str
    audio_path: str
    text: str
    duration_s: float


def load_audio(path: str, target_sr: int = TARGET_SR) -> np.ndarray:
    audio, sr = sf.read(path, dtype="float32", always_2d=False)
    if audio.ndim > 1:
        audio = audio.mean(axis=1)
    if sr != target_sr:
        import librosa

        audio = librosa.resample(audio, orig_sr=sr, target_sr=target_sr)
    return audio.astype(np.float32)


def iter_librispeech(root: str) -> Iterator[Utterance]:
    root_p = Path(root)
    if not root_p.exists():
        raise FileNotFoundError(root)

    for trans in sorted(root_p.rglob("*.trans.txt")):
        refs = {}
        with open(trans, encoding="utf-8") as f:
            for line in f:
                parts = line.strip().split(" ", 1)
                if len(parts) == 2:
                    refs[parts[0]] = parts[1]
        for uid, text in sorted(refs.items()):
            p = trans.parent / f"{uid}.flac"
            if not p.exists():
                continue
            info = sf.info(str(p))
            yield Utterance(uid, str(p), text, info.frames / info.samplerate)


def iter_manifest(path: str) -> Iterator[Utterance]:
    """JSONL per utterance: {"id":.., "audio":.., "text":..}. Entry point for any corpus."""
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            d = json.loads(line)
            ap = d["audio"]
            dur = d.get("duration_s")
            if dur is None:
                info = sf.info(ap)
                dur = info.frames / info.samplerate
            yield Utterance(str(d.get("id", Path(ap).stem)), ap, d["text"], float(dur))


def iter_utterances(source: str, path: str) -> Iterator[Utterance]:
    if source == "librispeech":
        return iter_librispeech(path)
    if source == "manifest":
        return iter_manifest(path)
    raise ValueError(f"unknown source: {source}")


In [ ]:
%%writefile /kaggle/working/code/src/campaign/__init__.py
"""Long-running data campaign: many models x many datasets x candidate composition.

Entry point is `python -m campaign.run --config run_config.json`; see config.py for the plan.
"""


In [ ]:
%%writefile /kaggle/working/code/src/campaign/config.py
"""What the campaign decodes, in which order, and where everything lives.

The plan is a flat list of jobs. A job is one model decoding one shard of one dataset under one
or more arms: decoding configurations that share a single pass of the audio encoder. Every job
carries a tier, and workers always take the lowest tier first. If the session ends early, every
cell of the design then has the same depth of coverage, instead of some cells being complete and
others empty.

Shards are consecutive slices of a dataset's fixed pseudo-random order (see prep_data.order_key),
so the first n utterances of any set are a random sample, and a model x set cell that reached
shard 2 holds exactly the same utterances for every model that reached it.
"""

from __future__ import annotations

import copy
import json
from pathlib import Path

# ---------------------------------------------------------------------------------------------
# datasets
# ---------------------------------------------------------------------------------------------

LS_REPO = "openslr/librispeech_asr"
ESB_REPO = "hf-audio/open-asr-leaderboard"

# kind: how prep_data builds it. n_max: cap after filtering (None = all).
SETS: dict[str, dict] = {
    # LibriSpeech. dev-* is only ever used to tune selection/ROVER parameters.
    "ls-dev-clean": dict(kind="hf_parquet", repo=LS_REPO, files=["all/validation.clean/0000.parquet"],
                         lang="en", role="dev", n_max=1000, domain="read audiobook"),
    "ls-dev-other": dict(kind="hf_parquet", repo=LS_REPO, files=["all/validation.other/0000.parquet"],
                         lang="en", role="dev", n_max=1000, domain="read audiobook, harder"),
    "ls-test-clean": dict(kind="hf_parquet", repo=LS_REPO, files=["all/test.clean/0000.parquet"],
                          lang="en", role="test", n_max=None, domain="read audiobook"),
    "ls-test-other": dict(kind="hf_parquet", repo=LS_REPO, files=["all/test.other/0000.parquet"],
                          lang="en", role="test", n_max=None, domain="read audiobook, harder"),
    # Open ASR Leaderboard test sets. Shards are spread over the file list because the files
    # may be sorted; `take` of `n_shards` are downloaded and sampled from.
    "ami": dict(kind="esb", config="ami", n_shards=15, take=4, lang="en", role="test",
                n_max=1600, domain="meetings, spontaneous"),
    "earnings22": dict(kind="esb", config="earnings22", n_shards=5, take=2, lang="en", role="test",
                       n_max=1600, domain="earnings calls, accented"),
    "voxpopuli": dict(kind="esb", config="voxpopuli", n_shards=4, take=2, lang="en", role="test",
                      n_max=1600, domain="parliament speeches"),
    "gigaspeech": dict(kind="esb", config="gigaspeech", n_shards=19, take=2, lang="en", role="test",
                       n_max=1600, domain="podcasts and YouTube"),
    "spgispeech": dict(kind="esb", config="spgispeech", n_shards=38, take=2, lang="en", role="test",
                       n_max=1600, domain="financial calls, formatted text"),
    "common_voice": dict(kind="esb", config="common_voice", n_shards=3, take=2, lang="en", role="test",
                         n_max=1600, domain="crowdsourced, many accents"),
    "fleurs-en": dict(kind="fleurs", code="en_us", lang="en", role="test", n_max=None,
                      domain="read Wikipedia sentences"),
    "slr83": dict(kind="slr83", subsets=["irish_english_male", "midlands_english_female",
                                         "northern_english_female", "scottish_english_female",
                                         "welsh_english_female"],
                  per_subset=320, lang="en", role="test", n_max=None, domain="UK and Irish accents"),
    # Controlled difficulty: the first n utterances of test-clean with added noise. The same
    # noise realisation is used at every SNR, so SNR is the only thing that changes.
    "ls-tc-babble10": dict(kind="noise", base="ls-test-clean", babble_from="ls-dev-other",
                           noise="babble", snr=10, n=600, lang="en", role="test", domain="babble 10 dB"),
    "ls-tc-babble5": dict(kind="noise", base="ls-test-clean", babble_from="ls-dev-other",
                          noise="babble", snr=5, n=600, lang="en", role="test", domain="babble 5 dB"),
    "ls-tc-babble0": dict(kind="noise", base="ls-test-clean", babble_from="ls-dev-other",
                          noise="babble", snr=0, n=600, lang="en", role="test", domain="babble 0 dB"),
    "ls-tc-white5": dict(kind="noise", base="ls-test-clean", babble_from="ls-dev-other",
                         noise="white", snr=5, n=600, lang="en", role="test", domain="white noise 5 dB"),
    # Multilingual, Drax only (Whisfusion and its tokenizer are English).
    "fleurs-de": dict(kind="fleurs", code="de_de", lang="de", role="test", n_max=800, domain="read, German"),
    "fleurs-fr": dict(kind="fleurs", code="fr_fr", lang="fr", role="test", n_max=800, domain="read, French"),
    "fleurs-es": dict(kind="fleurs", code="es_419", lang="es", role="test", n_max=800, domain="read, Spanish"),
    "fleurs-it": dict(kind="fleurs", code="it_it", lang="it", role="test", n_max=800, domain="read, Italian"),
    "fleurs-pt": dict(kind="fleurs", code="pt_br", lang="pt", role="test", n_max=800, domain="read, Portuguese"),
}

# order matters: prep_data builds them in this order, so early tiers can start sooner
PREP_ORDER = ["ls-dev-clean", "ls-dev-other", "ls-test-clean", "ls-test-other",
              "ls-tc-babble10", "ls-tc-babble5", "ls-tc-babble0", "ls-tc-white5",
              "fleurs-en", "ami", "earnings22", "voxpopuli", "common_voice", "gigaspeech",
              "spgispeech", "slr83", "fleurs-de", "fleurs-fr", "fleurs-es", "fleurs-it", "fleurs-pt"]

DEV_SETS = ["ls-dev-clean", "ls-dev-other"]
EN_EVAL = ["ls-test-clean", "ls-test-other", "ami", "earnings22", "voxpopuli", "gigaspeech",
           "spgispeech", "common_voice", "fleurs-en", "slr83",
           "ls-tc-babble10", "ls-tc-babble5", "ls-tc-babble0", "ls-tc-white5"]
MULTI_EVAL = ["fleurs-de", "fleurs-fr", "fleurs-es", "fleurs-it", "fleurs-pt"]
CONTROL_SETS = ["ls-dev-other", "ls-test-clean", "ls-test-other", "ami", "earnings22",
                "common_voice", "fleurs-en", "ls-tc-babble5"]

# utterances outside these bounds are dropped for every model, so all cells stay paired
MIN_DUR, MAX_DUR = 0.8, 28.0
MAX_REF_WORDS = 75          # Drax decodes into a fixed 128-token canvas

# ---------------------------------------------------------------------------------------------
# models
# ---------------------------------------------------------------------------------------------

MODELS: dict[str, dict] = {
    "whisfusion": dict(family="wf", paradigm="masked diffusion", langs=["en"],
                       encoder="whisper-small", params="170M decoder + 88M encoder"),
    "drax": dict(family="drax", hf="aiola/drax-v1", paradigm="discrete flow matching",
                 langs=["en", "de", "fr", "es", "it", "pt"], encoder="whisper-large-v3",
                 params="580M decoder + 635M encoder"),
    "whisper-small": dict(family="whisper", hf="openai/whisper-small", paradigm="autoregressive",
                          langs=["en"], encoder="whisper-small", params="244M"),
    "whisper-turbo": dict(family="whisper", hf="openai/whisper-large-v3-turbo",
                          paradigm="autoregressive", langs=["en"], encoder="whisper-large-v3",
                          params="809M"),
    "parakeet-ctc": dict(family="ctc", hf="nvidia/parakeet-ctc-1.1b", paradigm="CTC, one-step parallel",
                         langs=["en"], encoder="fastconformer", params="1.1B"),
}

DEFAULTS = {
    "mode": "full",
    "run_hours": 11.3,            # wall clock from start to the last output byte
    "tail_minutes": 30,           # reserved at the end for analysis and aggregation
    "shard_size": 200,
    "seed": 20260919,
    # Drax costs ~4x Whisfusion per candidate on a T4 (580M DiT, 16 steps, 51k vocab), so it gets
    # half the pool; the kscale jobs extend both ladders to 64 on one set
    "k_main": {"whisfusion": 32, "drax": 16},
    "k_scale": 64,
    "drax": {"T": 1.3, "steps": 16, "precision": "fp16", "ref_T": 0.1, "ref_K": 4},
    "drax_sweep_T": [0.1, 0.4, 0.7, 1.0, 1.3, 1.6, 2.0],
    "drax_sweep_K": 16,
    "drax_steps_sweep": [4, 8, 32],
    "whisper_sample": {"K": 16, "T": 0.6},
    "whisper_beams": 8,
    "ctc_sample": {"K": 32, "T": 1.0},
    "models": ["whisfusion", "drax", "whisper-small", "whisper-turbo", "parakeet-ctc"],
    "plan": "full",               # "round2" builds only what the first session missed
    "drax_low_T": 0.4,            # best realised temperature in the round-1 dev sweep
    "max_tier": 99,
    "analysis_workers": 2,
    # smoke-only knobs; ignored in full mode
    "smoke": {},
}


def load_config(path: str | None) -> dict:
    cfg = copy.deepcopy(DEFAULTS)
    if path:
        with open(path, encoding="utf-8") as f:
            user = json.load(f)
        for k, v in user.items():
            if isinstance(v, dict) and isinstance(cfg.get(k), dict):
                cfg[k].update(v)
            else:
                cfg[k] = v
    return cfg


def sets_for(cfg: dict) -> dict[str, dict]:
    """Dataset specs with any smoke-mode reductions applied."""
    sets = copy.deepcopy(SETS)
    sm = cfg.get("smoke") or {}
    if cfg["mode"] == "smoke":
        keep = sm.get("sets")
        if keep:
            sets = {k: v for k, v in sets.items() if k in keep or any(
                v2.get("base") == k or v2.get("babble_from") == k
                for k2, v2 in SETS.items() if k2 in keep)}
        n_cap = sm.get("n_max", 40)
        for name, s in sets.items():
            if s["kind"] == "esb":
                s["take"] = 1
            if s["kind"] == "slr83":
                s["subsets"] = s["subsets"][:1]
                s["per_subset"] = n_cap
            if s["kind"] == "noise":
                s["n"] = min(s["n"], n_cap)
            else:
                s["n_max"] = n_cap if s.get("n_max") is None else min(s["n_max"], n_cap)
    return sets


# ---------------------------------------------------------------------------------------------
# arms
# ---------------------------------------------------------------------------------------------

def arms_main(model: str, cfg: dict) -> list[dict]:
    fam = MODELS[model]["family"]
    if fam == "wf":
        return [dict(name="main", K=cfg["k_main"]["whisfusion"], steps=4,
                     schedule=[1.0, 0.9, 0.85, 0.8], seq_len=256)]
    if fam == "drax":
        d = cfg["drax"]
        return [dict(name="main", K=cfg["k_main"]["drax"], T=d["T"], steps=d["steps"]),
                dict(name="ref", K=d["ref_K"], T=d["ref_T"], steps=d["steps"])]
    if fam == "whisper":
        s = cfg["whisper_sample"]
        return [dict(name="greedy", mode="greedy"),
                dict(name="beam", mode="beam", beams=cfg["whisper_beams"]),
                dict(name="sample", mode="sample", K=s["K"], T=s["T"])]
    if fam == "ctc":
        s = cfg["ctc_sample"]
        return [dict(name="greedy", mode="greedy"), dict(name="sample", mode="sample", K=s["K"], T=s["T"])]
    raise ValueError(model)


def arms_drax_sweep(cfg: dict) -> list[dict]:
    d = cfg["drax"]
    return [dict(name=f"T{t:g}", K=cfg["drax_sweep_K"], T=t, steps=d["steps"])
            for t in cfg["drax_sweep_T"]]


def arms_drax_steps(cfg: dict) -> list[dict]:
    d = cfg["drax"]
    return [dict(name=f"steps{s}", K=cfg["drax_sweep_K"], T=d["T"], steps=s)
            for s in cfg["drax_steps_sweep"]] + [
            dict(name=f"steps{d['steps']}", K=cfg["drax_sweep_K"], T=d["T"], steps=d["steps"])]


def arms_kscale(model: str, cfg: dict) -> list[dict]:
    """One large pool, so the K ladder in the analysis extends to 64."""
    K = cfg.get("k_scale", 64)
    if MODELS[model]["family"] == "wf":
        return [dict(name=f"K{K}", K=K, steps=4, schedule=[1.0, 0.9, 0.85, 0.8], seq_len=256)]
    d = cfg["drax"]
    return [dict(name=f"K{K}", K=K, T=d["T"], steps=d["steps"])]


def arms_wf_ablation() -> list[dict]:
    base = dict(schedule=[1.0, 0.9, 0.85, 0.8], seq_len=256)
    return [dict(name="base16", K=16, steps=4, **base),
            dict(name="steps8", K=16, steps=8, **base),
            dict(name="fss_T1", K=16, steps=4, first_step_sampling=True, temperature=1.0, **base)]


# ---------------------------------------------------------------------------------------------
# plan
# ---------------------------------------------------------------------------------------------

def n_shards(set_name: str, n_utts: int, shard_size: int) -> int:
    return max(1, -(-n_utts // shard_size))


def expected_size(spec: dict) -> int:
    """Upper bound on a set's size before prep; the real count comes from its manifest."""
    if spec["kind"] == "noise":
        return spec["n"]
    if spec["kind"] == "slr83":
        return spec["per_subset"] * len(spec["subsets"])
    if spec.get("n_max"):
        return spec["n_max"]
    return {"ls-test-clean": 2620, "ls-test-other": 2939, "fleurs-en": 647}.get(spec.get("name", ""), 3000)


def build_plan(cfg: dict) -> list[dict]:
    """Every job, each with (tier, order). Lower is sooner."""
    sets = sets_for(cfg)
    for name, s in sets.items():
        s["name"] = name
    S = cfg["shard_size"]
    models = [m for m in cfg["models"] if m in MODELS]
    jobs: list[dict] = []

    def add(tier, model, set_name, shard, kind, arms, **extra):
        if set_name not in sets or model not in models:
            return
        lang = sets[set_name]["lang"]
        if lang not in MODELS[model]["langs"]:
            return
        jid = f"{model}__{set_name}__s{shard:02d}__{kind}"
        jobs.append(dict(id=jid, model=model, family=MODELS[model]["family"], set=set_name,
                         lang=lang, shard=shard, shard_size=S, kind=kind, arms=arms,
                         tier=tier, **extra))

    core = [m for m in ("drax", "whisfusion") if m in models]
    controls = [m for m in ("whisper-small", "whisper-turbo", "parakeet-ctc") if m in models]

    def max_shard(set_name):
        return n_shards(set_name, expected_size(sets[set_name]), S) if set_name in sets else 0

    # tier 0: dev, for tuning
    for sh in range(2):
        for s in DEV_SETS:
            for m in core:
                add(0, m, s, sh, "main", arms_main(m, cfg))
    # tiers 1, 2: the first two shards of every evaluation cell
    for tier, sh in [(1, 0), (2, 1)]:
        for s in EN_EVAL + MULTI_EVAL:
            for m in core:
                if sh < max_shard(s):
                    add(tier, m, s, sh, "main", arms_main(m, cfg))
    if "drax" in models:
        add(2, "drax", "ls-dev-other", 0, "sweepT", arms_drax_sweep(cfg))
    if "whisfusion" in models:
        add(2, "whisfusion", "ls-dev-other", 0, "ablation", arms_wf_ablation())
    # tier 3: controls on shard 0, K=64 scaling, and a third shard of every core cell
    for s in CONTROL_SETS:
        for m in controls:
            add(3, m, s, 0, "main", arms_main(m, cfg))
    for m in core:
        add(3, m, "ls-test-other", 0, "kscale", arms_kscale(m, cfg))
    for s in EN_EVAL + MULTI_EVAL:
        for m in core:
            if 2 < max_shard(s):
                add(3, m, s, 2, "main", arms_main(m, cfg))
    # tier 4: more ablations, controls shard 1, core shards 3-4
    if "drax" in models:
        add(4, "drax", "ami", 0, "sweepT", arms_drax_sweep(cfg))
        add(4, "drax", "ls-test-other", 0, "steps", arms_drax_steps(cfg))
    if "whisfusion" in models:
        add(4, "whisfusion", "ami", 0, "ablation", arms_wf_ablation())
    for s in CONTROL_SETS:
        for m in controls:
            if 1 < max_shard(s):
                add(4, m, s, 1, "main", arms_main(m, cfg))
    for sh in (3, 4):
        for s in EN_EVAL + MULTI_EVAL:
            for m in core:
                if sh < max_shard(s):
                    add(4, m, s, sh, "main", arms_main(m, cfg))
    # tier 5: everything else, round-robin over sets
    for sh in range(5, 20):
        for s in EN_EVAL + MULTI_EVAL:
            for m in core:
                if sh < max_shard(s):
                    add(5, m, s, sh, "main", arms_main(m, cfg))

    # smoke: one shard of everything, whatever the tier said
    if cfg["mode"] == "smoke":
        sm = cfg.get("smoke") or {}
        extra_sweep_shards = sm.get("drax_calibration_shards", 0)
        keep = []
        seen = set()
        for j in jobs:
            key = (j["model"], j["set"], j["kind"])
            if j["shard"] != 0 or key in seen:
                continue
            seen.add(key)
            keep.append(j)
        jobs = keep
        for sh in range(1, 1 + extra_sweep_shards):
            add(2, "drax", "ls-dev-other", sh, "sweepT", arms_drax_sweep(cfg))
        if sm.get("precision_check") and "drax" in models:
            add(1, "drax", "ls-dev-clean", 0, "prec-bf16", arms_main("drax", cfg), precision="bf16")

    jobs = [j for j in jobs if j["tier"] <= cfg["max_tier"]]
    # order: tier, then shard, then the listing order above (round-robin over sets and models)
    for i, j in enumerate(jobs):
        j["order"] = i
    jobs.sort(key=lambda j: (j["tier"], j["shard"], j["order"]))
    for i, j in enumerate(jobs):
        j["order"] = i
    return jobs


def build_plan_round2(cfg: dict) -> list[dict]:
    """Second session: what the first one could not reach.

    - the autoregressive and CTC controls, which never got a GPU turn in round 1;
    - Drax at the temperature its own dev sweep liked best (T=0.4), so composition can be
      reported at Drax's best operating point and not only at the diverse T=1.3;
    - the Whisfusion K=64 scaling job.

    Shards are the same utterances as round 1 (the ordering is a hash of the id), so everything
    stays paired with the data already collected.
    """
    sets = sets_for(cfg)
    for name, s in sets.items():
        s["name"] = name
    S = cfg["shard_size"]
    models = [m for m in cfg["models"] if m in MODELS]
    jobs: list[dict] = []

    def add(tier, model, set_name, shard, kind, arms):
        if set_name not in sets or model not in models:
            return
        if sets[set_name]["lang"] not in MODELS[model]["langs"]:
            return
        jobs.append(dict(id=f"{model}__{set_name}__s{shard:02d}__{kind}", model=model,
                         family=MODELS[model]["family"], set=set_name, lang=sets[set_name]["lang"],
                         shard=shard, shard_size=S, kind=kind, arms=arms, tier=tier))

    controls = [m for m in ("whisper-small", "whisper-turbo", "parakeet-ctc") if m in models]
    low = [dict(name="lowT", K=cfg["k_main"]["drax"], T=cfg.get("drax_low_T", 0.4),
                steps=cfg["drax"]["steps"])]

    for s in CONTROL_SETS:                                   # tier 0: the missing controls
        for m in controls:
            add(0, m, s, 0, "main", arms_main(m, cfg))
    for s in EN_EVAL + MULTI_EVAL:                           # tier 1: Drax at its best temperature
        add(1, "drax", s, 0, "lowT", low)
    add(1, "whisfusion", "ls-test-other", 0, "kscale", arms_kscale("whisfusion", cfg))
    for s in CONTROL_SETS:                                   # tier 2+: more of the same
        for m in controls:
            add(2, m, s, 1, "main", arms_main(m, cfg))
    for s in EN_EVAL + MULTI_EVAL:
        add(3, "drax", s, 1, "lowT", low)
    for sh in (2, 3):
        for s in CONTROL_SETS:
            for m in controls:
                add(4, m, s, sh, "main", arms_main(m, cfg))

    jobs = [j for j in jobs if j["tier"] <= cfg["max_tier"]]
    for i, j in enumerate(jobs):
        j["order"] = i
    jobs.sort(key=lambda j: (j["tier"], j["shard"], j["order"]))
    for i, j in enumerate(jobs):
        j["order"] = i
    return jobs


def model_key(job: dict) -> str:
    """Worker identity: one process per (model, precision)."""
    return job["model"] + (f"@{job['precision']}" if job.get("precision") else "")


def write_json(path: Path, obj) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)
    tmp.replace(path)


def build_plan_sweep(cfg: dict) -> list[dict]:
    """Third session: the temperature ladder on more than one dataset.

    Round 1 swept Drax's temperature on ls-dev-other only; the same job on ami sat at tier 4 and
    no worker reached it. One dataset cannot carry a claim about how composition scales with
    candidate disagreement. This plan runs nothing but the ladder, and puts two extra points
    between T=1.0 and T=1.6, where the whole transition happens and round 1 has no measurement.
    """
    sets = sets_for(cfg)
    for name, s in sets.items():
        s["name"] = name
    jobs: list[dict] = []
    for i, name in enumerate(cfg.get("sweep_sets", ["ami"])):
        if name not in sets or sets[name]["lang"] not in MODELS["drax"]["langs"]:
            continue
        jobs.append(dict(id=f"drax__{name}__s00__sweepT", model="drax",
                         family=MODELS["drax"]["family"], set=name, lang=sets[name]["lang"],
                         shard=0, shard_size=cfg["shard_size"], kind="sweepT",
                         arms=arms_drax_sweep(cfg), tier=0, order=i))
    return jobs


# ---------------------------------------------------------------------------------------------
# tree arms
# ---------------------------------------------------------------------------------------------

def arms_tree(cfg: dict) -> list[dict]:
    """Where to branch, by feature, at a mask budget where the choice can matter.

    Round 1 settled the shape question: sharing the early steps is 1.7x cheaper for nothing
    (tree-early) and 3x cheaper for about half a point (tree-deep), and none of it moved
    availability or recoverability. It also killed pure uncertainty targeting, which starved
    the confident positions of revision and cost 30 WER points.

    So this round asks the remaining question -- does it help to choose WHICH positions get
    re-predicted -- and asks it properly. Every targeted arm keeps part of the budget uniform
    (mask_mix), so exploration never stops, and mix = 0 is flat by construction. flat-sched
    is the same lowered schedule with no targeting, so the schedule is never the explanation.
    """
    K = cfg["k_main"]["whisfusion"]
    q = max(K // 4, 1)
    low = cfg.get("tree_low_schedule", [1.0, 0.5, 0.35, 0.25])
    mix = cfg.get("tree_mask_mix", 0.5)
    tree = [1, q, K, K]
    arms = [
        dict(name="flat", K=K, steps=4),
        dict(name="tree-early", K=K, steps=4, branch_schedule=tree),
        dict(name="flat-sched", K=K, steps=4, schedule=low),
    ]
    for mode in ("uncertain", "entropy", "margin", "unstable", "disagree"):
        arms.append(dict(name=f"aim-{mode}", K=K, steps=4, schedule=low,
                         mask_mode=mode, mask_mix=mix, branch_schedule=tree))
    # the mixture sweep, on the one feature round 1 already has a reading for
    for m in (0.25, 1.0):
        arms.append(dict(name=f"aim-uncertain-mix{m:g}", K=K, steps=4, schedule=low,
                         mask_mode="uncertain", mask_mix=m, branch_schedule=tree))
    return arms


def build_plan_tree(cfg: dict) -> list[dict]:
    """Nothing but the tree ablation on Whisfusion, on the sets named in tree_sets."""
    sets = sets_for(cfg)
    for name, s in sets.items():
        s["name"] = name
    S = cfg["shard_size"]
    jobs, order = [], 0
    names = [n for n in cfg.get("tree_sets", ["ls-test-other"]) if n in sets]

    def max_shard(name):
        return n_shards(name, expected_size(sets[name]), S)
    # Shard-major, so if the budget runs out every set still has the same depth.
    for shard in range(cfg.get("tree_shards", 1)):
        for name in names:
            if shard >= max_shard(name):
                continue
            jobs.append(dict(id=f"whisfusion__{name}__s{shard:02d}__tree", model="whisfusion",
                             family=MODELS["whisfusion"]["family"], set=name,
                             lang=sets[name]["lang"], shard=shard, shard_size=cfg["shard_size"],
                             kind="tree", arms=arms_tree(cfg), tier=0, order=order))
            order += 1
    return jobs


In [ ]:
%%writefile /kaggle/working/code/src/campaign/state.py
"""Filesystem state shared by the orchestrator and the workers.

    <root>/run_config.json   the resolved config
    <root>/plan.json         every job, sorted by (tier, order)
    <root>/state/claims/     <job>.claim, created with O_EXCL by the worker that takes it
    <root>/state/done/       <job>.json, written when a job ends (done, partial or failed)
    <root>/state/attempts/   crash counters
    <root>/state/failed_models/
    <root>/dumps/<job>/<arm>.jsonl.gz
    <data>/manifests/<set>.jsonl | <set>.failed

Nothing here is held in memory across processes; every question is answered from the files,
so a worker that dies leaves nothing inconsistent behind.
"""

from __future__ import annotations

import json
import os
import time
from pathlib import Path

from campaign import config as C

MAX_ATTEMPTS = 2


class State:
    def __init__(self, root: Path, data: Path):
        self.root = Path(root)
        self.data = Path(data)
        self.claims = self.root / "state" / "claims"
        self.done_dir = self.root / "state" / "done"
        self.attempts = self.root / "state" / "attempts"
        self.failed_models = self.root / "state" / "failed_models"
        self.dumps = self.root / "dumps"
        self.logs = self.root / "logs"
        self.analysis = self.root / "analysis"
        self.manifests = self.data / "manifests"
        for d in (self.claims, self.done_dir, self.attempts, self.failed_models, self.dumps,
                  self.logs, self.analysis, self.manifests):
            d.mkdir(parents=True, exist_ok=True)
        self._plan = None
        self._man: dict[str, list[dict]] = {}

    # --------------------------------------------------------------------------------- config

    def config(self) -> dict:
        with open(self.root / "run_config.json", encoding="utf-8") as f:
            return json.load(f)

    def plan(self) -> list[dict]:
        if self._plan is None:
            with open(self.root / "plan.json", encoding="utf-8") as f:
                self._plan = json.load(f)
        return self._plan

    def manifest(self, set_name: str) -> list[dict]:
        if set_name not in self._man:
            with open(self.manifests / f"{set_name}.jsonl", encoding="utf-8") as f:
                self._man[set_name] = [json.loads(line) for line in f if line.strip()]
        return self._man[set_name]

    # ---------------------------------------------------------------------------------- facts

    def manifest_ready(self, s: str) -> bool:
        return (self.manifests / f"{s}.jsonl").exists()

    def manifest_failed(self, s: str) -> bool:
        return (self.manifests / f"{s}.failed").exists()

    def prep_finished(self) -> bool:
        return (self.manifests / "_prep_done").exists()

    def is_done(self, jid: str) -> bool:
        return (self.done_dir / f"{jid}.json").exists()

    def is_claimed(self, jid: str) -> bool:
        return (self.claims / f"{jid}.claim").exists()

    def model_failed(self, mkey: str) -> bool:
        return (self.failed_models / f"{mkey}.txt").exists()

    def shard_empty(self, job: dict) -> bool:
        """Planned from an estimate of the set size; the manifest may be shorter."""
        return job["shard"] * job["shard_size"] >= len(self.manifest(job["set"]))

    def status(self, job: dict) -> str:
        """done | claimed | ready | waiting (for data) | dead (can never run)."""
        jid = job["id"]
        if self.is_done(jid):
            return "done"
        if self.is_claimed(jid):
            return "claimed"
        if self.model_failed(C.model_key(job)) or self.manifest_failed(job["set"]):
            return "dead"
        if not self.manifest_ready(job["set"]):
            return "dead" if self.prep_finished() else "waiting"
        if self.shard_empty(job):
            return "dead"
        return "ready"

    # ------------------------------------------------------------------------------- actions

    def claim_next(self, mkey: str, lane: int) -> dict | None:
        for job in self.plan():
            if C.model_key(job) != mkey or self.status(job) != "ready":
                continue
            try:
                fd = os.open(self.claims / f"{job['id']}.claim", os.O_CREAT | os.O_EXCL | os.O_WRONLY)
            except FileExistsError:
                continue
            with os.fdopen(fd, "w") as f:
                f.write(json.dumps({"lane": lane, "pid": os.getpid(), "t": time.time()}))
            return job
        return None

    def release(self, jid: str, crashed: bool = False) -> None:
        if crashed:
            p = self.attempts / f"{jid}.n"
            n = int(p.read_text()) + 1 if p.exists() else 1
            p.write_text(str(n))
            if n >= MAX_ATTEMPTS:
                self.mark_done(jid, {"job": jid, "status": "crashed", "attempts": n})
        (self.claims / f"{jid}.claim").unlink(missing_ok=True)

    def mark_done(self, jid: str, stats: dict) -> None:
        C.write_json(self.done_dir / f"{jid}.json", stats)

    def mark_model_failed(self, mkey: str, msg: str) -> None:
        (self.failed_models / f"{mkey}.txt").write_text(msg, encoding="utf-8")

    def orphaned_claims(self, lane: int) -> list[str]:
        """Claims held by a lane whose worker has exited, with no done marker."""
        out = []
        for p in self.claims.glob("*.claim"):
            jid = p.name[:-len(".claim")]
            if self.is_done(jid):
                continue
            try:
                info = json.loads(p.read_text())
            except Exception:
                continue
            if info.get("lane") == lane:
                out.append(jid)
        return out

    # ------------------------------------------------------------------------------ schedule

    def min_ready_tier(self) -> dict[str, int]:
        """Lowest tier with ready work, per model key."""
        out: dict[str, int] = {}
        for job in self.plan():
            k = C.model_key(job)
            if k in out and out[k] <= job["tier"]:
                continue
            if self.status(job) == "ready":
                out[k] = min(out.get(k, 99), job["tier"])
        return out

    def started(self) -> set[str]:
        """Model keys that have had their turn: a finished job, or one in flight on some lane.

        A model another lane is already running must count as started, or the two lanes keep
        yielding to each other and neither does any work.
        """
        out = set()
        for p in self.done_dir.glob("*.json"):
            try:
                s = json.loads(p.read_text())
            except Exception:
                continue
            if s.get("model"):
                out.add(s["model"] + (f"@{s['precision']}" if s.get("precision") else ""))
        by_id = {j["id"]: j for j in self.plan()}
        for p in self.claims.glob("*.claim"):
            job = by_id.get(p.name[: -len(".claim")])
            if job is not None:
                out.add(C.model_key(job))
        return out

    def should_yield(self, mkey: str) -> bool:
        """Give up the GPU when another model needs it more: it has work in a lower tier, or it
        has equal-tier work and has not run at all yet. Without the second rule a lane sticks
        with its model for the whole tier and a model late in the tier never starts."""
        tiers = self.min_ready_tier()
        mine = tiers.get(mkey)
        if mine is None:
            return False
        others = {k: t for k, t in tiers.items() if k != mkey}
        if not others:
            return False
        if min(others.values()) < mine:
            return True
        done = self.started()
        return mkey in done and any(t == mine and k not in done for k, t in others.items())

    def choose_model(self, current: str | None, preferred: list[str], busy: set[str]) -> str | None:
        """Model for a free lane: lowest ready tier first, then the lane's current model, then
        one no other lane is running, then the lane's preference order."""
        tiers = self.min_ready_tier()
        if not tiers:
            return None
        best = min(tiers.values())
        cands = [k for k, t in tiers.items() if t == best]
        # a model that has never run comes first, so every model is represented in a tier
        done = self.started()
        fresh = [k for k in cands if k not in done and k not in busy]
        if fresh:
            for p in preferred:
                if p in fresh:
                    return p
            return sorted(fresh)[0]
        if current in cands:
            return current
        free = [k for k in cands if k not in busy] or cands
        for p in preferred:
            if p in free:
                return p
        return sorted(free)[0]

    def any_waiting(self) -> bool:
        return any(self.status(j) == "waiting" for j in self.plan())

    def counts(self) -> dict:
        out: dict[str, int] = {}
        for j in self.plan():
            s = self.status(j)
            out[s] = out.get(s, 0) + 1
        return out


In [ ]:
%%writefile /kaggle/working/code/src/campaign/tokwords.py
"""Per-word confidences from per-token ones, for tokenizers that mark word starts.

Whisper's byte-level BPE marks a word start with a leading "Ġ" (an encoded space); SentencePiece
uses "▁". Returns None when the grouping does not line up with the normalised words, so a
confidence is never attached to the wrong word; ROVER then votes by frequency alone.
"""

from __future__ import annotations

WORD_MARKERS = ("Ġ", "▁")          # "Ġ", "▁"


def word_conf_from_ids(tokenizer, ids: list[int], confs: list[float], text: str) -> list[float] | None:
    from scorers import normalize

    if not ids:
        return None
    pieces = tokenizer.convert_ids_to_tokens(list(ids))
    groups: list[tuple[list[str], list[float]]] = []
    for piece, c in zip(pieces, confs):
        if piece is None:
            continue
        starts = piece.startswith(WORD_MARKERS)
        if starts or not groups:
            groups.append(([piece], [float(c)]))
        else:
            groups[-1][0].append(piece)
            groups[-1][1].append(float(c))

    out = []
    for toks, cs in groups:
        try:
            word = tokenizer.convert_tokens_to_string(toks)
        except Exception:
            word = "".join(toks)
        # one group can still hold several normalised words ("5,000" -> "5000" is one, but
        # "--" between two words glued without a space is not); repeat its confidence
        for _ in normalize(word).split():
            out.append(sum(cs) / len(cs))
    return out if len(out) == len(normalize(text).split()) else None


def token_stats(text: str, probs: list[float], logprobs: list[float], entropies: list[float] | None,
                word_conf: list[float] | None, extra: dict | None = None) -> dict:
    """A candidate in the dump schema from its emitted tokens' probabilities."""
    import statistics

    if not probs:
        d = dict(text=text, avg_conf=0.0, min_conf=0.0, median_conf=0.0, mean_logprob=-20.0,
                 mean_entropy=0.0, n_tokens=0)
    else:
        d = dict(text=text,
                 avg_conf=float(sum(probs) / len(probs)),
                 min_conf=float(min(probs)),
                 median_conf=float(statistics.median(probs)),
                 mean_logprob=float(sum(logprobs) / len(logprobs)),
                 mean_entropy=float(sum(entropies) / len(entropies)) if entropies else 0.0,
                 n_tokens=len(probs))
    if word_conf is not None:
        d["word_conf"] = word_conf
    if extra:
        d.update(extra)
    return d


In [ ]:
%%writefile /kaggle/working/code/src/campaign/prep_data.py
#!/usr/bin/env python3
"""Download every evaluation set and write it as a JSONL manifest of 16 kHz mono audio.

Each set is filtered to the same bounds (config.MIN_DUR..MAX_DUR, <= MAX_REF_WORDS words) and
ordered by a hash of the utterance id, so shard k of a set is the same utterances for every
model. A manifest appears atomically when its set is complete; a set that fails leaves
<set>.failed instead, and the jobs that need it are skipped rather than retried forever.

Runs as its own process next to the GPU workers; sets come out in config.PREP_ORDER.
"""

from __future__ import annotations

import argparse
import io
import json
import random
import re
import shutil
import subprocess
import sys
import tarfile
import time
import traceback
import zipfile
import zlib
from pathlib import Path

import numpy as np
import soundfile as sf

sys.path.insert(0, str(Path(__file__).resolve().parent.parent))

from campaign import config as C  # noqa: E402

SR = 16000
SALT = "campaign-v1"


def order_key(uid: str) -> int:
    return zlib.crc32(f"{SALT}|{uid}".encode("utf-8"))


def safe_name(uid: str) -> str:
    base = re.sub(r"[^A-Za-z0-9._-]", "_", uid)[:120]
    return f"{base}_{zlib.crc32(uid.encode('utf-8')) & 0xffffff:06x}"


def n_words(text: str) -> int:
    import scorers
    return len(scorers.normalize(text).split())


def keep(dur: float, text: str) -> bool:
    if not text or not (C.MIN_DUR <= dur <= C.MAX_DUR):
        return False
    return 1 <= n_words(text) <= C.MAX_REF_WORDS


def to_mono16k(audio: np.ndarray, sr: int) -> np.ndarray:
    if audio.ndim > 1:
        audio = audio.mean(axis=1)
    audio = audio.astype(np.float32)
    if sr != SR:
        import librosa
        audio = librosa.resample(audio, orig_sr=sr, target_sr=SR).astype(np.float32)
    return audio


def decode_bytes(b: bytes) -> tuple[np.ndarray, int]:
    try:
        a, sr = sf.read(io.BytesIO(b), dtype="float32", always_2d=False)
        return a, sr
    except Exception:
        # mp3 on an old libsndfile, or anything else soundfile refuses: let ffmpeg do it
        import tempfile
        with tempfile.NamedTemporaryFile(suffix=".bin", delete=False) as f:
            f.write(b)
            p = f.name
        try:
            out = subprocess.run(["ffmpeg", "-nostdin", "-loglevel", "error", "-i", p, "-f", "f32le",
                                  "-ac", "1", "-ar", str(SR), "-"], capture_output=True, check=True)
            return np.frombuffer(out.stdout, dtype=np.float32).copy(), SR
        finally:
            Path(p).unlink(missing_ok=True)


def write_flac(path: Path, audio: np.ndarray) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    peak = float(np.max(np.abs(audio))) if audio.size else 0.0
    if peak > 0.999:
        audio = audio * (0.999 / peak)
    sf.write(str(path), audio, SR, format="FLAC", subtype="PCM_16")


def finish(man_dir: Path, name: str, rows: list[dict], meta: dict) -> None:
    rows.sort(key=lambda r: order_key(r["id"]))
    for i, r in enumerate(rows):
        r["rank"] = i
    tmp = man_dir / f"{name}.jsonl.tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    meta.update(n_selected=len(rows), audio_min=round(sum(r["duration_s"] for r in rows) / 60, 2),
                n_clusters=len({r["cluster"] for r in rows}))
    C.write_json(man_dir / f"{name}.meta.json", meta)
    tmp.replace(man_dir / f"{name}.jsonl")
    print(f"[prep] {name}: {len(rows)} utts, {meta['audio_min']} min, "
          f"{meta['n_clusters']} clusters", flush=True)


def select(cands: list[dict], n_max: int | None) -> list[dict]:
    cands = [c for c in cands if keep(c["duration_s"], c["text"])]
    cands.sort(key=lambda c: order_key(c["id"]))
    return cands if n_max is None else cands[:n_max]


# ---------------------------------------------------------------------------------------------
# clusters: the unit a cluster bootstrap resamples. Speaker where known, else recording.
# ---------------------------------------------------------------------------------------------

AMI_MEETING = re.compile(r"((?:EN|ES|IS|TS|IB|IN)\d{4}[a-z])", re.I)


def cluster_of(set_name: str, uid: str, row: dict) -> str:
    if row.get("speaker_id") not in (None, ""):
        return str(row["speaker_id"])
    if set_name == "ami":
        m = AMI_MEETING.search(uid)
        if m:
            return m.group(1).upper()
    if set_name == "gigaspeech" and "_S" in uid:
        return uid.split("_S")[0]
    if set_name == "earnings22":
        return re.split(r"[_/-]", uid)[0]
    if set_name == "voxpopuli":
        return uid.split("_")[0]
    if set_name == "spgispeech" and "/" in uid:
        return uid.split("/")[0]
    if set_name == "common_voice":
        return uid              # no speaker ids in the leaderboard copy
    return uid


# ---------------------------------------------------------------------------------------------
# builders
# ---------------------------------------------------------------------------------------------

def hf_file(repo: str, filename: str, raw: Path) -> Path:
    from huggingface_hub import hf_hub_download
    for attempt in range(4):
        try:
            return Path(hf_hub_download(repo_id=repo, filename=filename, repo_type="dataset",
                                        local_dir=str(raw / repo.replace("/", "__"))))
        except Exception as e:
            print(f"[prep] download {repo}/{filename} failed ({type(e).__name__}: {e}), retry",
                  flush=True)
            time.sleep(5 * (attempt + 1))
    raise RuntimeError(f"cannot download {repo}/{filename}")


def build_parquet(name: str, spec: dict, files: list[Path], audio_dir: Path) -> tuple[list[dict], dict]:
    import pyarrow.parquet as pq

    cands = []
    audio_by_id: dict[str, bytes] = {}
    n_total = 0
    for fp in files:
        t = pq.read_table(str(fp))
        cols = t.column_names
        id_col = "id" if "id" in cols else ("file" if "file" in cols else cols[0])
        text_col = next(c for c in ("text", "transcription", "sentence", "normalized_text") if c in cols)
        ids = [str(x) for x in t.column(id_col).to_pylist()]
        texts = t.column(text_col).to_pylist()
        durs = t.column("audio_length_s").to_pylist() if "audio_length_s" in cols else None
        spk = t.column("speaker_id").to_pylist() if "speaker_id" in cols else None
        audio = t.column("audio").to_pylist()
        n_total += len(ids)
        for i, uid in enumerate(ids):
            b = audio[i]["bytes"] if isinstance(audio[i], dict) else None
            if not b:
                continue
            if durs is not None and durs[i] is not None:
                dur = float(durs[i])
            else:
                try:
                    info = sf.info(io.BytesIO(b))
                    dur = info.frames / info.samplerate
                except Exception:
                    continue
            row = {"speaker_id": spk[i] if spk else None}
            cands.append(dict(id=uid, text=str(texts[i] or ""), duration_s=round(dur, 3),
                              cluster=cluster_of(name, uid, row)))
            audio_by_id[uid] = b
        del t, audio

    chosen = select(cands, spec.get("n_max"))
    rows = []
    for c in chosen:
        try:
            a, sr = decode_bytes(audio_by_id[c["id"]])
            a = to_mono16k(a, sr)
        except Exception as e:
            print(f"[prep] {name}/{c['id']}: undecodable ({type(e).__name__})", flush=True)
            continue
        p = audio_dir / name / f"{safe_name(c['id'])}.flac"
        write_flac(p, a)
        rows.append(dict(c, audio=str(p), lang=spec["lang"], set=name,
                         duration_s=round(len(a) / SR, 3)))
    meta = dict(source=spec.get("repo", C.ESB_REPO), files=[str(f.name) for f in files],
                n_total=n_total, n_passing_filter=len([c for c in cands if keep(c["duration_s"], c["text"])]))
    return rows, meta


def build_esb(name, spec, raw, audio_dir):
    n = spec["n_shards"]
    idx = sorted({min(n - 1, int((i + 0.5) * n / spec["take"])) for i in range(spec["take"])})
    files = [hf_file(C.ESB_REPO, f"{spec['config']}/test-{i:05d}-of-{n:05d}.parquet", raw) for i in idx]
    rows, meta = build_parquet(name, spec, files, audio_dir)
    meta["shards_used"] = idx
    for f in files:
        f.unlink(missing_ok=True)
    return rows, meta


def build_ls(name, spec, raw, audio_dir):
    files = [hf_file(spec["repo"], fn, raw) for fn in spec["files"]]
    rows, meta = build_parquet(name, spec, files, audio_dir)
    return rows, meta


def build_fleurs(name, spec, raw, audio_dir):
    from huggingface_hub import hf_hub_download
    repo = "google/fleurs"
    d = raw / f"fleurs_{spec['code']}"
    tsv = Path(hf_hub_download(repo, f"data/{spec['code']}/test.tsv", repo_type="dataset", local_dir=str(d)))
    tar = Path(hf_hub_download(repo, f"data/{spec['code']}/audio/test.tar.gz", repo_type="dataset",
                               local_dir=str(d)))
    cands = []
    with open(tsv, encoding="utf-8") as f:
        for line in f:
            p = line.rstrip("\n").split("\t")
            if len(p) < 6:
                continue
            sent_id, fname, raw_text = p[0], p[1], p[2]
            try:
                dur = int(p[5]) / SR
            except ValueError:
                continue
            uid = fname.rsplit(".", 1)[0]
            cands.append(dict(id=uid, text=raw_text, duration_s=round(dur, 3), cluster=f"sent{sent_id}",
                              _fname=fname))
    chosen = {c["_fname"]: c for c in select(cands, spec.get("n_max"))}
    rows = []
    with tarfile.open(tar, "r:gz") as tf:
        for m in tf:
            base = m.name.rsplit("/", 1)[-1]
            if base not in chosen or not m.isfile():
                continue
            b = tf.extractfile(m).read()
            a, sr = decode_bytes(b)
            a = to_mono16k(a, sr)
            c = chosen[base]
            p = audio_dir / name / f"{safe_name(c['id'])}.flac"
            write_flac(p, a)
            rows.append(dict({k: v for k, v in c.items() if not k.startswith("_")}, audio=str(p),
                             lang=spec["lang"], set=name, duration_s=round(len(a) / SR, 3)))
    shutil.rmtree(d, ignore_errors=True)
    return rows, dict(source=repo, code=spec["code"], n_total=len(cands))


SLR83_MIRRORS = ["https://www.openslr.org/resources/83/{}.zip",
                 "https://us.openslr.org/resources/83/{}.zip",
                 "https://openslr.elda.org/resources/83/{}.zip",
                 "https://openslr.trmal.net/resources/83/{}.zip"]


def curl(urls: list[str], out: Path) -> None:
    out.parent.mkdir(parents=True, exist_ok=True)
    for url in urls:
        r = subprocess.run(["curl", "-L", "--fail", "--retry", "3", "--connect-timeout", "20",
                            "-s", "-o", str(out), url])
        if r.returncode == 0 and out.exists() and out.stat().st_size > 1000:
            return
        print(f"[prep] {url} failed (curl {r.returncode})", flush=True)
    raise RuntimeError(f"all mirrors failed for {out.name}")


def build_slr83(name, spec, raw, audio_dir):
    rows, per = [], {}
    for sub in spec["subsets"]:
        z = raw / "slr83" / f"{sub}.zip"
        if not z.exists():
            curl([u.format(sub) for u in SLR83_MIRRORS], z)
        cands = []
        with zipfile.ZipFile(z) as zf:
            names = {n.rsplit("/", 1)[-1]: n for n in zf.namelist()}
            index = names.get("line_index.csv")
            if index is None:
                raise RuntimeError(f"{sub}: no line_index.csv")
            for line in zf.read(index).decode("utf-8").splitlines():
                parts = line.strip().split(",", 2)
                if len(parts) != 3:
                    continue
                file_id, text = parts[1].strip(), parts[2].strip()
                if f"{file_id}.wav" not in names:
                    continue
                cands.append(dict(id=file_id, text=text, _member=names[f"{file_id}.wav"],
                                  cluster="_".join(file_id.split("_")[:2])))
            # duration needs the header, so read candidates in order until enough pass
            cands.sort(key=lambda c: order_key(c["id"]))
            got = 0
            for c in cands:
                if got >= spec["per_subset"]:
                    break
                b = zf.read(c["_member"])
                info = sf.info(io.BytesIO(b))
                dur = info.frames / info.samplerate
                if not keep(dur, c["text"]):
                    continue
                a, sr = decode_bytes(b)
                a = to_mono16k(a, sr)
                p = audio_dir / name / f"{safe_name(c['id'])}.flac"
                write_flac(p, a)
                rows.append(dict(id=c["id"], text=c["text"], cluster=c["cluster"], audio=str(p),
                                 lang=spec["lang"], set=name, duration_s=round(len(a) / SR, 3),
                                 subset=sub))
                got += 1
        per[sub] = got
        z.unlink(missing_ok=True)
        print(f"[prep] slr83/{sub}: {got}", flush=True)
    return rows, dict(source="openslr SLR83", per_subset=per)


def read_manifest(path: Path) -> list[dict]:
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


def build_noise(name, spec, raw, audio_dir, man_dir: Path):
    base = read_manifest(man_dir / f"{spec['base']}.jsonl")[:spec["n"]]
    pool = read_manifest(man_dir / f"{spec['babble_from']}.jsonl")
    snr = float(spec["snr"])
    rows = []
    cache: dict[str, np.ndarray] = {}

    def load(p):
        if p not in cache:
            a, sr = sf.read(p, dtype="float32")
            cache[p] = to_mono16k(a, sr)
        return cache[p]

    for r in base:
        speech = load(r["audio"])
        n = len(speech)
        # seeded by the utterance alone, so every SNR of one utterance shares its noise
        rng = np.random.default_rng(order_key(r["id"]))
        if spec["noise"] == "babble":
            noise = np.zeros(n, dtype=np.float32)
            for j in rng.choice(len(pool), size=min(6, len(pool)), replace=False):
                src = load(pool[int(j)]["audio"])
                src = src / (np.sqrt(np.mean(src ** 2)) + 1e-8)
                reps = int(np.ceil((n + len(src)) / len(src)))
                tiled = np.tile(src, reps)
                off = int(rng.integers(0, len(src)))
                noise += tiled[off:off + n]
        else:
            noise = rng.standard_normal(n).astype(np.float32)
        ps = float(np.mean(speech ** 2)) + 1e-12
        pn = float(np.mean(noise ** 2)) + 1e-12
        mix = speech + noise * np.sqrt(ps / (pn * 10 ** (snr / 10.0)))
        p = audio_dir / name / f"{safe_name(r['id'])}.flac"
        write_flac(p, mix.astype(np.float32))
        rows.append(dict(id=r["id"], text=r["text"], cluster=r["cluster"], audio=str(p),
                         lang=r["lang"], set=name, duration_s=r["duration_s"]))
    if len(cache) > 2000:
        cache.clear()
    return rows, dict(source=f"{spec['base']} + {spec['noise']} @ {snr:g} dB SNR",
                      babble_from=spec["babble_from"])


def main() -> int:
    ap = argparse.ArgumentParser()
    ap.add_argument("--config", default=None)
    ap.add_argument("--data", required=True)
    ap.add_argument("--only", nargs="*", default=None)
    args = ap.parse_args()

    cfg = C.load_config(args.config)
    sets = C.sets_for(cfg)
    data = Path(args.data)
    raw, audio_dir, man_dir = data / "raw", data / "audio", data / "manifests"
    man_dir.mkdir(parents=True, exist_ok=True)

    todo = [s for s in C.PREP_ORDER if s in sets and (args.only is None or s in args.only)]
    t_all = time.time()
    for name in todo:
        spec = sets[name]
        if (man_dir / f"{name}.jsonl").exists():
            print(f"[prep] {name}: already there", flush=True)
            continue
        t0 = time.time()
        try:
            kind = spec["kind"]
            if kind == "hf_parquet":
                rows, meta = build_ls(name, spec, raw, audio_dir)
            elif kind == "esb":
                rows, meta = build_esb(name, spec, raw, audio_dir)
            elif kind == "fleurs":
                rows, meta = build_fleurs(name, spec, raw, audio_dir)
            elif kind == "slr83":
                rows, meta = build_slr83(name, spec, raw, audio_dir)
            elif kind == "noise":
                for dep in (spec["base"], spec["babble_from"]):
                    if not (man_dir / f"{dep}.jsonl").exists():
                        raise RuntimeError(f"needs {dep}, which is not prepared")
                rows, meta = build_noise(name, spec, raw, audio_dir, man_dir)
            else:
                raise ValueError(kind)
            if not rows:
                raise RuntimeError("no utterances survived")
            meta.update(set=name, kind=kind, lang=spec["lang"], role=spec["role"],
                        domain=spec.get("domain"), filter=dict(min_dur=C.MIN_DUR, max_dur=C.MAX_DUR,
                                                               max_ref_words=C.MAX_REF_WORDS),
                        prep_s=round(time.time() - t0, 1))
            finish(man_dir, name, rows, meta)
        except Exception as e:
            msg = f"{type(e).__name__}: {e}"
            print(f"[prep] {name} FAILED: {msg}", flush=True)
            traceback.print_exc()
            (man_dir / f"{name}.failed").write_text(msg + "\n" + traceback.format_exc(), encoding="utf-8")
    shutil.rmtree(raw, ignore_errors=True)
    print(f"[prep] all done in {(time.time() - t_all) / 60:.1f} min", flush=True)
    (man_dir / "_prep_done").write_text(str(time.time()), encoding="utf-8")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


In [ ]:
%%writefile /kaggle/working/code/src/campaign/drax_fast.py
"""Drax with the audio encoded once, K candidates in one batch, and per-token confidences.

The public Transcriber encodes the same file once per candidate, in fp32, and returns only text.
This keeps Drax's own modules and weights and replaces only the driver:

- the Whisper encoder and the cross-attention K/V cache run once per utterance and are expanded
  over the K candidates;
- the DiT blocks run under fp16 autocast on Turing (bf16 is emulated there), with the embeddings,
  time embedding, output layer and residual stream kept in fp32 so large activations cannot
  overflow;
- the sampler is DraxMixtureDiscreteEulerSolver with PolynomialConvexScheduler(n=1), written out
  so the one-hot velocity is never materialised: with kappa_t = t, a position jumps to its
  sampled x_1 with probability 1 - exp(-h / (1 - t)) unless it already holds it, which is what
  the reference solver's categorical over u reduces to.

The drax package is imported without running drax/__init__.py, which would pull in torchaudio and
flow_matching for the Transcriber this module does not use.
"""

from __future__ import annotations

import importlib
import json
import math
import sys
import types
from contextlib import nullcontext
from pathlib import Path

import torch
import torch.nn.functional as F

from campaign.tokwords import word_conf_from_ids


def import_drax(drax_root: str):
    """drax.model.* without drax/__init__.py."""
    pkg_dir = Path(drax_root) / "drax"
    if "drax" not in sys.modules:
        pkg = types.ModuleType("drax")
        pkg.__path__ = [str(pkg_dir)]
        sys.modules["drax"] = pkg
    return importlib.import_module("drax.model.transformer")


def _sinusoids(length: int, channels: int, max_timescale: float = 10000) -> torch.Tensor:
    log_inc = math.log(max_timescale) / (channels // 2 - 1)
    inv = torch.exp(-log_inc * torch.arange(channels // 2))
    t = torch.arange(length)[:, None] * inv[None, :]
    return torch.cat([t.sin(), t.cos()], dim=1)


def _build_encoder(wcfg: dict):
    from transformers import WhisperConfig
    from transformers.models.whisper.modeling_whisper import WhisperEncoder

    conf = WhisperConfig(**{k: v for k, v in wcfg.items() if k not in ("torch_dtype", "transformers_version")})
    try:
        with torch.device("meta"):
            enc = WhisperEncoder(conf)
        return enc.to_empty(device="cpu")
    except Exception:
        return WhisperEncoder(conf)


class DraxFast:
    def __init__(self, model_id: str, drax_root: str, device: str = "cuda", precision: str = "fp16",
                 verbose: bool = True):
        from huggingface_hub import hf_hub_download
        from safetensors.torch import load_file
        from transformers import WhisperFeatureExtractor, WhisperTokenizer

        tr = import_drax(drax_root)
        self.device = device
        self.precision = precision if device == "cuda" else "fp32"
        self.block_dtype = {"fp16": torch.float16, "bf16": torch.bfloat16, "fp32": torch.float32}[self.precision]

        with open(hf_hub_download(model_id, "config.json"), encoding="utf-8") as f:
            cfg = json.load(f)
        dcfg, wcfg = cfg["decoder_config"], cfg["whisper_config"]

        enc = _build_encoder(wcfg)
        enc_state = load_file(hf_hub_download(model_id, "encoder.safetensors"))
        res = enc.load_state_dict(enc_state, strict=False)
        missing = [k for k in res.missing_keys if "embed_positions" not in k]
        if missing or res.unexpected_keys:
            raise RuntimeError(f"Drax encoder keys: missing {missing[:5]} unexpected {res.unexpected_keys[:5]}")
        if not any(k.endswith("embed_positions.weight") for k in enc_state):
            with torch.no_grad():
                enc.embed_positions.weight.copy_(_sinusoids(*enc.embed_positions.weight.shape))
        del enc_state
        enc_dtype = torch.float16 if device == "cuda" else torch.float32
        self.encoder = enc.to(device=device, dtype=enc_dtype).train(False).requires_grad_(False)

        dec = tr.Transformer(vocab_size=dcfg["vocab_size"], masked=dcfg.get("masked", False), config=dcfg)
        dec_state = load_file(hf_hub_download(model_id, "decoder.safetensors"))
        res = dec.load_state_dict(dec_state, strict=False)
        missing = [k for k in res.missing_keys if "inv_freq" not in k]
        if missing or res.unexpected_keys:
            raise RuntimeError(f"Drax decoder keys: missing {missing[:5]} unexpected {res.unexpected_keys[:5]}")
        del dec_state
        dec = dec.to(device).train(False).requires_grad_(False)
        if self.block_dtype != torch.float32:
            for blk in dec.blocks:
                blk.to(self.block_dtype)
        self.dec = dec
        self.vocab = dcfg["vocab_size"]
        self.length = dcfg["length"]

        self.fe = WhisperFeatureExtractor.from_pretrained("openai/whisper-large-v3")
        self.tok = WhisperTokenizer.from_pretrained("openai/whisper-large-v3")
        self.eot = self.tok.convert_tokens_to_ids("<|endoftext|>")
        self.sot = self.tok.convert_tokens_to_ids("<|startoftranscript|>")
        self.transcribe_id = self.tok.convert_tokens_to_ids("<|transcribe|>")
        self.notimestamps = self.tok.convert_tokens_to_ids("<|notimestamps|>")
        self.langs = dcfg.get("support_language_codes", ["en"])
        if verbose:
            n_dec = sum(p.numel() for p in dec.parameters()) / 1e6
            n_enc = sum(p.numel() for p in self.encoder.parameters()) / 1e6
            print(f"[drax] decoder {n_dec:.0f}M ({self.precision} blocks), encoder {n_enc:.0f}M, "
                  f"canvas {self.length}, vocab {self.vocab}", flush=True)

    # -----------------------------------------------------------------------------------------

    def _ac(self, dtype):
        if self.device != "cuda" or dtype == torch.float32:
            return nullcontext()
        return torch.autocast("cuda", dtype=dtype)

    @torch.no_grad()
    def encode(self, audio, lang: str = "en") -> dict:
        feats = self.fe(audio, sampling_rate=16000, return_tensors="pt").input_features
        feats = feats.to(self.device, dtype=next(self.encoder.parameters()).dtype)
        h = self.encoder(feats).last_hidden_state                                # (1, 1500, 1280)
        if self.block_dtype == torch.float32:
            h = h.float()
        with self._ac(self.block_dtype):
            proj = self.dec.audio_proj(h)
            ks, vs = [], []
            for blk in self.dec.blocks:
                na = blk.norm_audio(proj)
                ks.append(blk.k_cross(na))
                vs.append(blk.v_cross(na))
        dt = self.block_dtype
        return {"k": [k.to(dt) for k in ks], "v": [v.to(dt) for v in vs]}

    def _prompt(self, lang: str) -> list[int]:
        return [self.sot, self.tok.convert_tokens_to_ids(f"<|{lang}|>"), self.transcribe_id,
                self.notimestamps]

    def _logits(self, x_t, t, preserve, cache) -> torch.Tensor:
        K = x_t.shape[0]
        d = self.dec
        x = d.vocab_embed(x_t) + d.preserve_embeddings(preserve.long())          # fp32 residual
        c = F.silu(d.time_embedding(time=t))
        rot = d.rotary_emb(x=x)
        with self._ac(self.block_dtype):
            for i, blk in enumerate(d.blocks):
                x = blk(x=x, rotary_cos_sin=rot, c=c, audio=None,
                        audio_k=cache["k"][i].expand(K, -1, -1), audio_v=cache["v"][i].expand(K, -1, -1))
        head = torch.float16 if self.block_dtype != torch.float32 else torch.float32
        with self._ac(head):
            out = d.output_layer(x=x.float(), c=c)
        return out.float()

    @torch.no_grad()
    def sample(self, cache: dict, lang: str, K: int, temperature: float, steps: int, seed: int,
               chunk: int = 32) -> list[dict]:
        """K candidates in the dump schema. Chunks bound memory at large K."""
        cands: list[dict] = []
        for start in range(0, K, chunk):
            cands += self._sample(cache, lang, min(chunk, K - start), temperature, steps,
                                  seed + 7919 * start)
        return cands

    def _sample(self, cache, lang, K, temperature, steps, seed) -> list[dict]:
        dev = self.device
        g = torch.Generator(device=dev)
        g.manual_seed(int(seed) & 0x7FFFFFFF)
        L, V = self.length, self.vocab
        prompt = torch.tensor(self._prompt(lang), device=dev)
        P = len(prompt)

        x_init = torch.randint(0, V, (K, L), device=dev, generator=g)
        x_init[:, :P] = prompt
        preserve = torch.zeros((K, L), dtype=torch.bool, device=dev)
        preserve[:, :P] = True

        h = 1.0 / steps if steps > 1 else 1.0 - 1e-8
        n_steps = math.ceil(1.0 / h - 1e-9)
        grid = [h * i for i in range(n_steps)] + [1.0]
        x_t = x_init.clone()
        final_logits = None
        inv_T = 1.0 / max(float(temperature), 1e-6)
        for i in range(n_steps):
            t, dt = grid[i], grid[i + 1] - grid[i]
            logits = self._logits(x_t, torch.full((K,), t, device=dev), preserve, cache)
            # Gumbel-max is categorical(softmax(logits / T)) with an explicit generator
            gum = torch.rand(logits.shape, device=dev, generator=g)
            gum.clamp_(min=1e-12).log_().neg_().log_().neg_()
            gum.add_(logits, alpha=inv_T)
            x_1 = gum.argmax(dim=-1)
            del gum
            x_1 = torch.where(preserve, x_init, x_1)
            x_t = torch.where(preserve, x_init, x_t)
            if i == n_steps - 1:
                x_t = x_1
                final_logits = logits
            else:
                p_jump = 1.0 - math.exp(-dt / (1.0 - t))
                jump = (torch.rand((K, L), device=dev, generator=g) < p_jump) & (x_1 != x_t)
                x_t = torch.where(jump, x_1, x_t)
                del logits

        if not bool(torch.isfinite(final_logits).all()):
            raise FloatingPointError(f"non-finite Drax logits under {self.precision}")
        logp = torch.log_softmax(final_logits, dim=-1)
        del final_logits
        chosen_lp = logp.gather(-1, x_t.unsqueeze(-1)).squeeze(-1)                 # (K, L)
        ent = -(logp.exp() * logp).sum(-1)
        del logp

        toks = x_t.cpu()
        chosen_lp, ent = chosen_lp.float().cpu(), ent.float().cpu()
        texts = self.tok.batch_decode(toks[:, P:], skip_special_tokens=True)
        out = []
        for i in range(K):
            ids = toks[i, P:]
            text_pos = (ids < self.eot).nonzero().squeeze(-1)
            text = texts[i].strip()
            if text_pos.numel() == 0:
                out.append(dict(text=text, avg_conf=0.0, min_conf=0.0, median_conf=0.0,
                                mean_logprob=-20.0, mean_entropy=0.0, n_tokens=0))
                continue
            lp = chosen_lp[i, P:][text_pos]
            pr = lp.exp()
            tid = ids[text_pos].tolist()
            d = dict(text=text,
                     avg_conf=float(pr.mean()),          # probability of the token actually emitted
                     min_conf=float(pr.min()),
                     median_conf=float(pr.median()),
                     mean_logprob=float(lp.mean()),
                     mean_entropy=float(ent[i, P:][text_pos].mean()),
                     n_tokens=len(tid))
            wc = word_conf_from_ids(self.tok, tid, pr.tolist(), text)
            if wc is not None:
                d["word_conf"] = wc
            out.append(d)
        return out


In [ ]:
%%writefile /kaggle/working/code/src/campaign/families.py
"""One adapter per model family, all with the same two calls.

    enc = fam.encode(audio, lang)            # once per utterance
    cands, extra = fam.decode(enc, arm, seed, lang)

Candidates are dicts in the dump_candidates.py schema (text, avg_conf, min_conf, median_conf,
mean_logprob, mean_entropy, n_tokens, optional word_conf), so scorers.py and compose.py work on
every family unchanged. `extra` is per-row metadata such as identical_after_step1.
"""

from __future__ import annotations

import os
from pathlib import Path

import numpy as np
import torch

from campaign.tokwords import token_stats, word_conf_from_ids


def _from_pretrained(cls, name, dtype, **kw):
    try:
        return cls.from_pretrained(name, dtype=dtype, **kw)
    except TypeError:
        return cls.from_pretrained(name, torch_dtype=dtype, **kw)


def _device() -> str:
    return "cuda" if torch.cuda.is_available() else "cpu"


# ---------------------------------------------------------------------------------------------

class Whisfusion:
    """Masked diffusion (Whisfusion v1): PDD with K parallel candidates, as decode.py does it."""

    def __init__(self, spec: dict, precision: str | None = None):
        from huggingface_hub import hf_hub_download

        import wf_model

        base = hf_hub_download("nieshen/SMDM", "mdm_safetensors/mdm-170M-100e18-rsl-0.01.safetensors")
        adapter = hf_hub_download("taeyoun811/whisfusion", "whisfusion_stage2_decoder.pt")
        self.wf = wf_model.load(base, adapter, device=_device(), dtype=precision or "auto")

    def encode(self, audio, lang):
        import wf_model
        return wf_model.encode_audio(self.wf, audio)

    def decode(self, enc, arm, seed, lang):
        import decode as dec

        r = dec.pdd_decode(self.wf, enc, n_candidates=arm["K"], n_steps=arm.get("steps", 4),
                           mask_ratio_schedule=arm.get("schedule"), seq_len=arm.get("seq_len", 256),
                           first_step_sampling=arm.get("first_step_sampling", False),
                           temperature=arm.get("temperature", 1.0), seed=seed,
                           branch_schedule=arm.get("branch_schedule"),
                           mask_mode=arm.get("mask_mode", "uniform"),
                           mask_mix=arm.get("mask_mix", 1.0),
                           adaptive=arm.get("adaptive"))
        return ([dec.candidate_to_dict(c) for c in r.candidates],
                {"identical_after_step1": r.identical_after_step1,
                 "n_candidates_used": r.n_candidates_used,
                 "uncertainty": r.uncertainty,
                 "branch_widths": r.branch_widths})


# ---------------------------------------------------------------------------------------------

class Drax:
    """Discrete flow matching (Drax-v1) through campaign.drax_fast."""

    def __init__(self, spec: dict, precision: str | None = None):
        from campaign.drax_fast import DraxFast

        root = os.environ.get("DRAX_ROOT", "/kaggle/working/drax_repo")
        self.m = DraxFast(spec["hf"], root, device=_device(), precision=precision or "fp16")

    def encode(self, audio, lang):
        return self.m.encode(audio, lang)

    def decode(self, enc, arm, seed, lang):
        chunk = int(os.environ.get("DRAX_CHUNK", "32"))
        return self.m.sample(enc, lang, K=arm["K"], temperature=arm["T"], steps=arm["steps"],
                             seed=seed, chunk=chunk), {}


# ---------------------------------------------------------------------------------------------

class Whisper:
    """Autoregressive control: greedy, beam n-best, or K temperature samples from one model.

    Calls GenerationMixin.generate directly. Whisper's own generate() turns
    num_return_sequences into repeated inputs, so beam search would return B copies of the best
    beam instead of an n-best list. Token probabilities come from one teacher-forced pass over
    the finished sequences, which works the same for every mode and transformers version.
    """

    def __init__(self, spec: dict, precision: str | None = None):
        import copy

        from transformers import WhisperForConditionalGeneration, WhisperProcessor

        self.dev = _device()
        self.dtype = torch.float16 if self.dev == "cuda" else torch.float32
        self.proc = WhisperProcessor.from_pretrained(spec["hf"])
        self.model = _from_pretrained(WhisperForConditionalGeneration, spec["hf"], self.dtype)
        self.model = self.model.to(self.dev).train(False).requires_grad_(False)
        self.tok = self.proc.tokenizer
        self.eot = self.tok.convert_tokens_to_ids("<|endoftext|>")
        self.gen_cfg = copy.deepcopy(self.model.generation_config)
        for k in ("forced_decoder_ids", "max_length"):
            if hasattr(self.gen_cfg, k):
                setattr(self.gen_cfg, k, None)

    def encode(self, audio, lang):
        f = self.proc.feature_extractor(audio, sampling_rate=16000, return_tensors="pt").input_features
        return f.to(self.dev, dtype=self.dtype)

    def _prompt(self, lang):
        ids = [self.tok.convert_tokens_to_ids(t) for t in
               ("<|startoftranscript|>", f"<|{lang}|>", "<|transcribe|>", "<|notimestamps|>")]
        return torch.tensor([ids], device=self.dev)

    @torch.no_grad()
    def decode(self, enc, arm, seed, lang):
        from transformers.generation.utils import GenerationMixin

        torch.manual_seed(int(seed) & 0x7FFFFFFF)
        prompt = self._prompt(lang)
        P = prompt.shape[1]
        kw = dict(input_features=enc, decoder_input_ids=prompt, generation_config=self.gen_cfg,
                  max_new_tokens=200)
        mode = arm["mode"]
        if mode == "greedy":
            seqs = GenerationMixin.generate(self.model, **kw, do_sample=False, num_beams=1)
        elif mode == "beam":
            B = arm["beams"]
            seqs = GenerationMixin.generate(self.model, **kw, do_sample=False, num_beams=B,
                                            num_return_sequences=B)
        else:
            seqs = GenerationMixin.generate(self.model, **kw, do_sample=True, temperature=arm["T"],
                                            top_k=0, top_p=1.0, num_return_sequences=arm["K"])
        if not torch.is_tensor(seqs):
            seqs = seqs.sequences
        N = seqs.shape[0]

        # teacher-forced pass for the probability of every emitted token; timed apart, since a
        # production decoder would get these from generate() for free
        import time
        if self.dev == "cuda":
            torch.cuda.synchronize()
        t_score = time.time()
        enc_h = self.model.model.encoder(enc).last_hidden_state
        logits = self.model(encoder_outputs=(enc_h.expand(N, -1, -1),), decoder_input_ids=seqs).logits
        lps = torch.log_softmax(logits.float(), dim=-1)[:, :-1]                   # predicts seqs[:, 1:]
        tgt = seqs[:, 1:]
        tok_lp = lps.gather(-1, tgt.unsqueeze(-1)).squeeze(-1).cpu()
        tok_ent = (-(lps.exp() * lps).sum(-1)).cpu()
        seqs = seqs.cpu()
        score_s = time.time() - t_score

        cands = []
        for i in range(N):
            gen = seqs[i, P:].tolist()
            if self.eot in gen:
                gen = gen[:gen.index(self.eot)]
            ids = [t for t in gen if t < self.eot]
            pos = [P - 1 + j for j, t in enumerate(gen) if t < self.eot]
            text = self.tok.decode(ids, skip_special_tokens=True).strip()
            lp = [float(tok_lp[i, p]) for p in pos]
            pr = [float(np.exp(x)) for x in lp]
            ent = [float(tok_ent[i, p]) for p in pos]
            wc = word_conf_from_ids(self.tok, ids, pr, text) if ids else None
            cands.append(token_stats(text, pr, lp, ent, wc, {"rank": i}))
        return cands, {"score_s": round(score_s, 4)}


# ---------------------------------------------------------------------------------------------

class CTC:
    """Parallel one-step control: greedy or K paths sampled from the CTC frame posteriors."""

    def __init__(self, spec: dict, precision: str | None = None):
        from transformers import AutoModelForCTC, AutoProcessor

        self.dev = _device()
        self.dtype = torch.float16 if self.dev == "cuda" else torch.float32
        self.proc = AutoProcessor.from_pretrained(spec["hf"])
        self.model = _from_pretrained(AutoModelForCTC, spec["hf"], self.dtype)
        self.model = self.model.to(self.dev).train(False).requires_grad_(False)
        self.tok = getattr(self.proc, "tokenizer", self.proc)
        pad = getattr(self.model.config, "pad_token_id", None)
        self.blank = int(pad) if pad is not None else None

    @torch.no_grad()
    def encode(self, audio, lang):
        inputs = self.proc(audio, sampling_rate=16000, return_tensors="pt")
        inputs = {k: (v.to(self.dev, dtype=self.dtype) if torch.is_floating_point(v) else v.to(self.dev))
                  for k, v in inputs.items() if hasattr(v, "to")}
        logits = self.model(**inputs).logits[0].float()
        if self.blank is None:
            self.blank = logits.shape[-1] - 1
        return torch.log_softmax(logits, dim=-1)                           # (T, V)

    def _collapse(self, path: torch.Tensor, lp: torch.Tensor):
        ids, confs, prev = [], [], None
        run: list[float] = []
        for t, tok in enumerate(path.tolist()):
            if tok != prev:
                if run:
                    confs.append(sum(run) / len(run))
                run = []
                if tok != self.blank:
                    ids.append(tok)
                prev = tok
            if tok != self.blank:
                run.append(float(lp[t, tok].exp()))
        if run:
            confs.append(sum(run) / len(run))
        return ids, confs

    @torch.no_grad()
    def decode(self, enc, arm, seed, lang):
        lp = enc
        if arm["mode"] == "greedy":
            paths = lp.argmax(-1, keepdim=True).T                           # (1, T)
        else:
            g = torch.Generator(device=lp.device)
            g.manual_seed(int(seed) & 0x7FFFFFFF)
            K = arm["K"]
            u = torch.rand((K,) + tuple(lp.shape), device=lp.device, generator=g).clamp_(min=1e-12)
            paths = (lp.unsqueeze(0) / arm["T"] - torch.log(-torch.log(u))).argmax(-1)   # (K, T)
        lp_cpu = lp.cpu()
        cands = []
        for p in paths.cpu():
            ids, confs = self._collapse(p, lp_cpu)
            try:
                text = self.tok.decode(ids, skip_special_tokens=True, group_tokens=False)
            except TypeError:
                text = self.tok.decode(ids, skip_special_tokens=True)
            text = text.strip()
            logp = [float(np.log(max(c, 1e-12))) for c in confs]
            wc = word_conf_from_ids(self.tok, ids, confs, text) if ids else None
            cands.append(token_stats(text, confs, logp, None, wc))
        return cands, {}


FAMILIES = {"wf": Whisfusion, "drax": Drax, "whisper": Whisper, "ctc": CTC}


def load(model_name: str, precision: str | None = None):
    from campaign.config import MODELS

    spec = MODELS[model_name]
    return FAMILIES[spec["family"]](spec, precision)


In [ ]:
%%writefile /kaggle/working/code/src/campaign/worker.py
#!/usr/bin/env python3
"""One GPU, one model: claim that model's jobs in plan order and decode them.

A claim is a file created with O_EXCL, so two workers never take the same job. Rows are
appended to a plain JSONL per (job, arm) as they are decoded and gzipped when the job ends; a
job cut short by the deadline or a crash keeps every finished utterance, and a rerun of the
same job skips them.

The worker exits instead of idling when (a) its model has nothing claimable left, or (b) some
other model has claimable work in a strictly lower tier: the parent then gives the GPU to that
model. Model load failures are recorded so no lane tries that model again.
"""

from __future__ import annotations

import argparse
import gzip
import json
import os
import queue
import shutil
import socket
import sys
import threading
import time
import traceback
import zlib
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent.parent))

from campaign import config as C  # noqa: E402
from campaign import state as ST  # noqa: E402


def round_floats(obj, nd=4):
    if isinstance(obj, float):
        return round(obj, nd)
    if isinstance(obj, list):
        return [round_floats(x, nd) for x in obj]
    if isinstance(obj, dict):
        return {k: round_floats(v, nd) for k, v in obj.items()}
    return obj


def utt_seed(base: int, uid: str, arm: str) -> int:
    return (base ^ zlib.crc32(f"{uid}|{arm}".encode("utf-8"))) & 0x7FFFFFFF


def read_done_ids(path: Path) -> set[str]:
    done = set()
    for p in (path, path.with_suffix(path.suffix + ".gz")):
        if not p.exists():
            continue
        opener = gzip.open if p.suffix == ".gz" else open
        try:
            with opener(p, "rt", encoding="utf-8") as f:
                for line in f:
                    try:
                        done.add(json.loads(line)["id"])
                    except Exception:
                        pass
        except (EOFError, OSError):
            pass
    return done


def compress(path: Path) -> None:
    """Merge a plain JSONL into its .gz (a resumed job may already have one)."""
    if not path.exists():
        return
    gz = path.with_suffix(path.suffix + ".gz")
    rows = []
    if gz.exists():
        try:
            with gzip.open(gz, "rt", encoding="utf-8") as f:
                rows = [line for line in f if line.strip()]
        except (EOFError, OSError):
            pass
    with open(path, encoding="utf-8") as f:
        rows += [line for line in f if line.strip()]
    tmp = gz.with_suffix(".tmp")
    with gzip.open(tmp, "wt", encoding="utf-8") as f:
        for line in rows:
            f.write(line if line.endswith("\n") else line + "\n")
    tmp.replace(gz)
    path.unlink()


class Prefetch:
    """Loads audio for the next utterances on a thread while the GPU decodes."""

    def __init__(self, rows, depth=6):
        import data as dataio

        self.q: queue.Queue = queue.Queue(maxsize=depth)
        self.rows = rows
        self._load = dataio.load_audio
        self.t = threading.Thread(target=self._run, daemon=True)
        self.t.start()

    def _run(self):
        for r in self.rows:
            try:
                self.q.put((r, self._load(r["audio"]), None))
            except Exception as e:
                self.q.put((r, None, e))
        self.q.put(None)

    def __iter__(self):
        while True:
            item = self.q.get()
            if item is None:
                return
            yield item


def run_job(job: dict, fam, st: ST.State, deadline: float, cfg: dict, log) -> dict:
    import torch

    man = st.manifest(job["set"])
    S = job["shard_size"]
    rows = man[job["shard"] * S:(job["shard"] + 1) * S]
    out_dir = st.dumps / job["id"]
    out_dir.mkdir(parents=True, exist_ok=True)
    paths = {a["name"]: out_dir / f"{a['name']}.jsonl" for a in job["arms"]}
    done = None
    for a in job["arms"]:
        d = read_done_ids(paths[a["name"]])
        done = d if done is None else done & d
    todo = [r for r in rows if r["id"] not in (done or set())]
    log(f"job {job['id']}: {len(rows)} utts, {len(rows) - len(todo)} already done, arms "
        f"{[a['name'] for a in job['arms']]}")

    cuda = torch.cuda.is_available()
    if cuda:
        torch.cuda.reset_peak_memory_stats()

    def sync():
        if cuda:
            torch.cuda.synchronize()

    stats = dict(n_ok=0, n_err=0, audio_s=0.0, encode_s=0.0, decode_s={a["name"]: 0.0 for a in job["arms"]},
                 started=time.time(), status="done")
    fhs = {k: open(p, "a", encoding="utf-8") for k, p in paths.items()}
    try:
        for n, (u, audio, err) in enumerate(Prefetch(todo)):
            if time.time() > deadline:
                stats["status"] = "partial"
                log(f"deadline reached inside {job['id']} after {n} utts")
                break
            if err is not None or audio is None:
                stats["n_err"] += 1
                log(f"  [!] {u['id']}: audio load failed: {err}")
                continue
            try:
                sync(); t0 = time.time()
                enc = fam.encode(audio, job["lang"])
                sync(); t_enc = time.time() - t0
                outs = {}
                for a in job["arms"]:
                    sync(); t1 = time.time()
                    cands, extra = fam.decode(enc, a, utt_seed(cfg["seed"], u["id"], a["name"]), job["lang"])
                    sync()
                    # time spent only on measurement (Whisper's confidence pass) is not decoding
                    outs[a["name"]] = (cands, extra, time.time() - t1 - extra.get("score_s", 0.0))
            except torch.cuda.OutOfMemoryError as e:  # type: ignore[attr-defined]
                stats["n_err"] += 1
                log(f"  [!] {u['id']}: CUDA OOM ({e}); skipping")
                torch.cuda.empty_cache()
                continue
            except Exception as e:
                stats["n_err"] += 1
                log(f"  [!] {u['id']}: {type(e).__name__}: {e}")
                if stats["n_err"] <= 3:
                    log(traceback.format_exc())
                if stats["n_err"] >= 20 and stats["n_err"] > 0.5 * (stats["n_ok"] + stats["n_err"]):
                    stats["status"] = "failed"
                    log(f"  too many errors in {job['id']}, giving up on it")
                    break
                continue

            for name, (cands, extra, t_dec) in outs.items():
                row = dict(id=u["id"], dataset=job["set"], lang=job["lang"], cluster=u.get("cluster"),
                           duration_s=u["duration_s"], reference=u["text"], model=job["model"],
                           arm=name, n_unique=len({c["text"] for c in cands}),
                           encode_s=round(t_enc, 4), decode_s=round(t_dec, 4), candidates=cands, **extra)
                fhs[name].write(json.dumps(round_floats(row), ensure_ascii=False) + "\n")
                stats["decode_s"][name] += t_dec
            stats["encode_s"] += t_enc
            stats["audio_s"] += u["duration_s"]
            stats["n_ok"] += 1
            if stats["n_ok"] % 10 == 0:
                for fh in fhs.values():
                    fh.flush()
            if stats["n_ok"] % 50 == 0:
                el = time.time() - stats["started"]
                log(f"  {job['id']}: {stats['n_ok']}/{len(todo)}  {el / stats['n_ok']:.2f} s/utt")
    finally:
        for fh in fhs.values():
            fh.close()
    for p in paths.values():
        compress(p)

    stats["wall_s"] = time.time() - stats["started"]
    stats["n_in_shard"] = len(rows)
    stats["n_resumed"] = len(rows) - len(todo)
    if cuda:
        stats["peak_mem_mb"] = torch.cuda.max_memory_allocated() / 2 ** 20
    return stats


def main() -> int:
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", required=True, help="model key, e.g. drax or drax@bf16")
    ap.add_argument("--lane", type=int, default=0)
    ap.add_argument("--root", required=True, help="campaign output dir")
    ap.add_argument("--data", required=True)
    ap.add_argument("--deadline", type=float, required=True)
    args = ap.parse_args()

    st = ST.State(Path(args.root), Path(args.data))
    cfg = st.config()
    model, _, precision = args.model.partition("@")
    log_path = st.logs / f"lane{args.lane}.log"

    def log(msg):
        line = f"[{time.strftime('%H:%M:%S')}] [lane{args.lane} {args.model}] {msg}"
        print(line, flush=True)
        with open(log_path, "a", encoding="utf-8") as f:
            f.write(line + "\n")

    import torch
    log(f"host {socket.gethostname()}  cuda={torch.cuda.is_available()}  "
        f"device={torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'}")

    fam = None
    n_jobs = 0
    while time.time() < args.deadline:
        if st.should_yield(args.model):
            log("yielding: another model has lower-tier work waiting")
            break
        job = st.claim_next(args.model, args.lane)
        if job is None:
            log("nothing claimable for this model")
            break
        if fam is None:
            t0 = time.time()
            try:
                from campaign import families
                fam = families.load(model, precision or None)
            except Exception as e:
                log(f"MODEL LOAD FAILED: {type(e).__name__}: {e}\n{traceback.format_exc()}")
                st.mark_model_failed(args.model, f"{type(e).__name__}: {e}")
                st.release(job["id"])
                return 3
            log(f"model loaded in {time.time() - t0:.0f} s")
        try:
            stats = run_job(job, fam, st, args.deadline, cfg, log)
        except Exception as e:
            log(f"JOB CRASHED {job['id']}: {type(e).__name__}: {e}\n{traceback.format_exc()}")
            st.release(job["id"], crashed=True)
            continue
        st.mark_done(job["id"], dict(stats, job=job["id"], model=job["model"], lane=args.lane))
        n_jobs += 1
        el = stats["wall_s"]
        log(f"done {job['id']} [{stats['status']}] {stats['n_ok']} utts in {el / 60:.1f} min "
            f"({el / max(stats['n_ok'], 1):.2f} s/utt, RTF {el / max(stats['audio_s'], 1e-6):.3f})")
    log(f"exiting after {n_jobs} jobs")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


In [ ]:
%%writefile /kaggle/working/code/src/campaign/analysis.py
#!/usr/bin/env python3
"""Per-utterance metrics for one dump (one job x arm), along a ladder of candidate counts.

Candidates in a dump are exchangeable draws, so the first k of them are a faithful simulation of
having decoded only k (see allocate.py). For every utterance and every k on the ladder this
records the edit count of each method, so any corpus WER, paired test or regression can be
computed later from integers without touching the candidates again:

    first          candidate 0: one decode, no selection at all
    conf, minconf, logprob     pick the candidate with the best model score
    mbr            pick the most central candidate (minimum Bayes risk under WER)
    cm05, cm15     conf + lambda * mbr, lambda 0.5 and 1.5
    rover_freq     ROVER, one vote per candidate
    rover_c        ROVER mixing vote share and word confidence (alpha 0.5, eps 0.7)
    rover_cg       the same with near-duplicate candidates sharing a vote (gamma 0.5)
    oracle_cand    best single candidate (needs the reference)
    oracle_comp    best path through the confusion network (needs the reference)
    control        oracle_comp with every alternative replaced by an unrelated word
    anti           worst single candidate

wn_* are the same edits after Whisper's text normaliser (the Open ASR Leaderboard convention),
for the main methods. A second table holds a parameter grid at the main k, for tuning on dev
and applying on test without re-running anything.
"""

from __future__ import annotations

import argparse
import gzip
import json
import random
import statistics
import sys
import time
import zlib
from pathlib import Path

from rapidfuzz.distance import Levenshtein

sys.path.insert(0, str(Path(__file__).resolve().parent.parent))

import compose  # noqa: E402
import scorers as S  # noqa: E402

LADDER = [1, 2, 3, 4, 5, 6, 8, 10, 12, 15, 16, 20, 24, 32, 48, 64]
ORACLE_KS = {2, 4, 8, 15, 16, 32, 64}
WN_KS = {1, 15, 16}
GRID_ALPHA = [0.3, 0.5, 0.7, 0.85, 1.0]
GRID_EPS = [0.3, 0.5, 0.7]
GRID_GAMMA = [0.0, 0.5, 1.0]
LAMBDAS = [0.0, 0.1, 0.25, 0.5, 0.75, 1.0, 1.5, 2.0, 3.0]
ROVER_DEFAULT = dict(alpha=0.5, eps=0.7, gamma=0.5)


# ------------------------------------------------------------------------------ normalisers

class WhisperNorm:
    """Whisper's English normaliser for en, the basic one otherwise; None if unavailable."""

    def __init__(self):
        self.en = self.basic = None
        try:
            from whisper_normalizer.basic import BasicTextNormalizer
            from whisper_normalizer.english import EnglishTextNormalizer
            self.en, self.basic = EnglishTextNormalizer(), BasicTextNormalizer()
            return
        except Exception:
            pass
        try:  # the same code ships inside transformers; it needs the spelling map of a tokenizer
            from transformers import WhisperTokenizer
            from transformers.models.whisper.english_normalizer import (BasicTextNormalizer,
                                                                        EnglishTextNormalizer)
            tok = WhisperTokenizer.from_pretrained("openai/whisper-small")
            self.en = EnglishTextNormalizer(tok.english_spelling_normalizer)
            self.basic = BasicTextNormalizer()
        except Exception:
            pass

    @property
    def ok(self) -> bool:
        return self.en is not None

    def __call__(self, text: str, lang: str) -> list[str]:
        f = self.en if lang == "en" else self.basic
        return f(text).split()


# ---------------------------------------------------------------------------------- loading

def read_dump(path: Path) -> list[dict]:
    rows = []
    opener = gzip.open if str(path).endswith(".gz") else open
    try:
        with opener(path, "rt", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    rows.append(json.loads(line))
                except json.JSONDecodeError:
                    pass
    except (EOFError, OSError):
        pass
    return rows


def prepare(row: dict) -> tuple[list[list[str]], list[list[float]], float]:
    """Word lists and per-word confidences. A candidate without usable word confidences gets
    its own mean confidence on every word, so confidence-weighted voting stays defined."""
    cands, confs, exact = [], [], 0
    for c in row["candidates"]:
        words = S.normalize(c["text"]).split()
        wc = c.get("word_conf")
        if wc is not None and len(wc) == len(words):
            confs.append([float(x) for x in wc])
            exact += 1
        else:
            a = float(c.get("avg_conf", 0.5))
            a = min(max(a, 0.0), 1.0)
            confs.append([a] * len(words))
        cands.append(words)
    return cands, confs, exact / max(len(cands), 1)


# --------------------------------------------------------------------------------- analysis

def argbest(values: list[float]) -> int:
    best, bi = None, 0
    for i, v in enumerate(values):
        if best is None or v > best:
            best, bi = v, i
    return bi


def analyze_row(row: dict, k_main: int, pool: list[str], wn: WhisperNorm, meta: dict):
    ref = S.normalize(row["reference"]).split()
    R = len(ref)
    lang = row.get("lang", "en")
    cands, confs, conf_frac = prepare(row)
    K = len(cands)
    if K == 0:
        return [], []
    raw = row["candidates"]
    ed = [Levenshtein.distance(ref, c) for c in cands]
    lens = [len(c) for c in cands]
    Lm = [[0] * K for _ in range(K)]
    for i in range(K):
        for j in range(i + 1, K):
            d = Levenshtein.distance(cands[i], cands[j])
            Lm[i][j] = Lm[j][i] = d
    avg = [float(c.get("avg_conf", 0.0)) for c in raw]
    mnc = [float(c.get("min_conf", 0.0)) for c in raw]
    mlp = [float(c.get("mean_logprob", 0.0)) for c in raw]

    use_wn = wn.ok
    if use_wn:
        ref_wn = wn(row["reference"], lang)
        cand_wn = [wn(c["text"], lang) for c in raw]
        ed_wn = [Levenshtein.distance(ref_wn, c) for c in cand_wn]
    rng = random.Random(zlib.crc32(row["id"].encode("utf-8")))

    base = dict(meta, id=row["id"], cluster=row.get("cluster") or row["id"],
                duration_s=row.get("duration_s"), ref_len=R, ref_len_wn=len(ref_wn) if use_wn else None,
                K_avail=K, conf_frac=round(conf_frac, 3), encode_s=row.get("encode_s"),
                decode_s=row.get("decode_s"), n_unique_all=row.get("n_unique"),
                identical_after_step1=row.get("identical_after_step1"))

    out, grid = [], []
    ks = [k for k in LADDER if k <= K]
    if K not in ks:
        ks.append(K)
    for k in ks:
        idx = range(k)
        if k > 1:
            mbr = [-sum(Lm[i][j] / max(lens[j], 1) for j in idx if j != i) / (k - 1) for i in idx]
        else:
            mbr = [0.0]
        p_conf = argbest(avg[:k])
        p_min = argbest(mnc[:k])
        p_lp = argbest(mlp[:k])
        p_mbr = argbest(mbr)
        p_cm05 = argbest([avg[i] + 0.5 * mbr[i] for i in idx])
        p_cm15 = argbest([avg[i] + 1.5 * mbr[i] for i in idx])
        sub, sub_c = cands[:k], confs[:k]

        t0 = time.perf_counter()
        slots1 = compose.confusion_network(sub, p_mbr, sub_c, None)
        w05 = compose.cluster_weights(sub, 0.5)
        slots05 = compose.confusion_network(sub, p_mbr, sub_c, w05)
        h_cg = compose.rover(slots05, sum(w05), ROVER_DEFAULT["alpha"], ROVER_DEFAULT["eps"])
        rover_ms = (time.perf_counter() - t0) * 1000.0
        h_freq = compose.rover(slots1, float(k), 1.0, 0.5)
        h_c = compose.rover(slots1, float(k), ROVER_DEFAULT["alpha"], ROVER_DEFAULT["eps"])

        rec = dict(base, k=k,
                   n_unique=len({" ".join(c) for c in sub}),
                   pairwise=round(100.0 * statistics.fmean(
                       Lm[i][j] / max(lens[i], lens[j], 1) for i in idx for j in idx if i < j), 3)
                   if k > 1 else 0.0,
                   e_first=ed[0], e_conf=ed[p_conf], e_minconf=ed[p_min], e_logprob=ed[p_lp],
                   e_mbr=ed[p_mbr], e_cm05=ed[p_cm05], e_cm15=ed[p_cm15],
                   e_rover_freq=Levenshtein.distance(ref, h_freq),
                   e_rover_c=Levenshtein.distance(ref, h_c),
                   e_rover_cg=Levenshtein.distance(ref, h_cg),
                   e_oracle_cand=min(ed[:k]), e_anti=max(ed[:k]),
                   e_oracle_comp=None, e_control=None, rover_ms=round(rover_ms, 3),
                   len_rover_cg=len(h_cg), len_conf=lens[p_conf])
        if k >= 2 and (k in ORACLE_KS or k == K):
            oc = compose.oracle_path(slots1, ref)
            ctrl = compose.oracle_path(compose.shuffled_control(slots1, pool, rng), ref)
            rec["e_oracle_comp"] = min(oc, rec["e_oracle_cand"])
            rec["e_control"] = min(ctrl, rec["e_oracle_cand"])
        if use_wn and (k in WN_KS or k == k_main or k == K):
            # Build the network from Whisper-normalised candidates rather than normalising
            # ROVER's legacy-normalised output a second time. That second pass cost 2-7 points
            # of pure artifact: legacy strips the apostrophe, so "time\'s leisure" becomes
            # "times leisure" and stays that way, while the reference becomes "time is leisure".
            # Per-word confidences cannot come along -- they index the model\'s own tokens and
            # this normaliser changes the word count -- so wn_rover_freq is the clean metric
            # here and wn_rover_c / _cg fall back to frequency with a flat confidence.
            sub_wn = cand_wn[:k]
            n1 = compose.confusion_network(sub_wn, p_mbr, None, None)
            w5 = compose.cluster_weights(sub_wn, 0.5)
            n5 = compose.confusion_network(sub_wn, p_mbr, None, w5)
            rec.update(wn_first=ed_wn[0], wn_conf=ed_wn[p_conf], wn_mbr=ed_wn[p_mbr],
                       wn_cm05=ed_wn[p_cm05],
                       wn_rover_freq=Levenshtein.distance(
                           ref_wn, compose.rover(n1, float(k), 1.0, 0.5)),
                       wn_rover_c=Levenshtein.distance(
                           ref_wn, compose.rover(n1, float(k), ROVER_DEFAULT["alpha"],
                                                 ROVER_DEFAULT["eps"])),
                       wn_rover_cg=Levenshtein.distance(
                           ref_wn, compose.rover(n5, sum(w5), ROVER_DEFAULT["alpha"],
                                                 ROVER_DEFAULT["eps"])),
                       wn_oracle_cand=min(ed_wn[:k]))
        out.append(rec)

        if k == min(k_main, K):
            gbase = dict(meta, id=row["id"], cluster=base["cluster"], ref_len=R, k=k)
            nets = {1.0: (slots1, float(k)), 0.5: (slots05, sum(w05))}
            for g in GRID_GAMMA:
                if g not in nets:
                    w = compose.cluster_weights(sub, g)
                    nets[g] = (compose.confusion_network(sub, p_mbr, sub_c, w), sum(w))
                slots, tot = nets[g]
                for a in GRID_ALPHA:
                    for e in (GRID_EPS if a < 1.0 else [0.5]):
                        h = compose.rover(slots, tot, a, e)
                        grid.append(dict(gbase, param="rover", alpha=a, eps=e, gamma=g, lam=None,
                                         e=Levenshtein.distance(ref, h)))
            for lam in LAMBDAS:
                p = argbest([avg[i] + lam * mbr[i] for i in idx])
                grid.append(dict(gbase, param="select", alpha=None, eps=None, gamma=None, lam=lam, e=ed[p]))
    return out, grid


def analyze_dump(path: Path, out_dir: Path, meta: dict, k_main: int) -> dict:
    import pandas as pd

    rows = read_dump(path)
    tag = meta["tag"]
    if not rows:
        return {"tag": tag, "n": 0}
    pool = [w for r in rows[:200] for w in S.normalize(r["candidates"][0]["text"]).split()] if rows[0]["candidates"] else []
    if not pool:
        pool = ["the"]
    wn = WhisperNorm()
    recs, grid = [], []
    t0 = time.time()
    for r in rows:
        try:
            a, g = analyze_row(r, k_main, pool, wn, meta)
        except Exception as e:  # one malformed row must not lose the dump
            print(f"[analysis] {tag}/{r.get('id')}: {type(e).__name__}: {e}", flush=True)
            continue
        recs += a
        grid += g
    out_dir.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(recs).to_parquet(out_dir / f"{tag}.parquet", index=False)
    if grid:
        pd.DataFrame(grid).to_parquet(out_dir / f"{tag}.grid.parquet", index=False)
    return {"tag": tag, "n": len(rows), "n_rec": len(recs), "seconds": round(time.time() - t0, 1),
            "wn": wn.ok}


def analyze_job(root: Path, job: dict, k_main: int) -> list[dict]:
    """Every arm of a finished job."""
    res = []
    for arm in job["arms"]:
        p = root / "dumps" / job["id"] / f"{arm['name']}.jsonl.gz"
        if not p.exists():
            p = p.with_suffix("")
            if not p.exists():
                continue
        meta = dict(tag=f"{job['id']}__{arm['name']}", job=job["id"], model=job["model"],
                    precision=job.get("precision") or "", set=job["set"], lang=job["lang"],
                    kind=job["kind"], arm=arm["name"], shard=job["shard"],
                    arm_K=arm.get("K", arm.get("beams", 1)), arm_T=arm.get("T", arm.get("temperature")),
                    arm_steps=arm.get("steps"), arm_mode=arm.get("mode"))
        km = min(k_main, arm.get("K", arm.get("beams", 1)))
        res.append(analyze_dump(p, root / "analysis", meta, km))
    return res


def main() -> int:
    ap = argparse.ArgumentParser()
    ap.add_argument("--root", required=True)
    ap.add_argument("--job", default=None, help="one job id; default: every finished job")
    ap.add_argument("--k_main", type=int, default=32)
    ap.add_argument("--redo", action="store_true")
    args = ap.parse_args()
    root = Path(args.root)
    with open(root / "plan.json", encoding="utf-8") as f:
        plan = {j["id"]: j for j in json.load(f)}
    jobs = [plan[args.job]] if args.job else [plan[p.stem] for p in (root / "state" / "done").glob("*.json")
                                                 if p.stem in plan]
    for job in jobs:
        if not args.redo and all((root / "analysis" / f"{job['id']}__{a['name']}.parquet").exists()
                                 for a in job["arms"]):
            continue
        for r in analyze_job(root, job, args.k_main):
            print(r, flush=True)
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


In [ ]:
%%writefile /kaggle/working/code/src/campaign/aggregate.py
#!/usr/bin/env python3
"""Everything the paper needs, from the per-utterance tables analysis.py wrote.

Outputs, under <root>/results/:

    per_utt_k.parquet, per_utt_k.csv.gz   one row per (model, set, arm, utterance, k): edits of
                                          every method; the source for any further statistics
    grid.parquet                          ROVER / selection parameter grid at the main k
    cells.csv                             corpus and mean-utterance WER of every method per
                                          (model, set, job kind, arm, k), with diversity
    contrasts.csv                         paired comparisons within an arm: delta, utterance and
                                          cluster bootstrap CIs, Wilcoxon p, wins / ties / losses
    cross_arm.csv                         composition against each model's standard decoding
                                          (Drax at low temperature, Whisper greedy / beam, CTC
                                          greedy, Whisfusion's own K=15 confidence pick)
    tuned.csv                             parameters picked on dev, reported on every test cell
    meta_analysis.csv                     random-effects pooling of the main contrast over sets
    timing.csv                            RTF, encoder / decoder split, ROVER CPU cost
    campaign_summary.json, REPORT.md

Deltas are WER(first) - WER(second) in points: positive means the second system is better.
"""

from __future__ import annotations

import argparse
import json
import math
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path(__file__).resolve().parent.parent))

from campaign import config as C  # noqa: E402

METHODS = ["first", "conf", "minconf", "logprob", "mbr", "cm05", "cm15", "rover_freq", "rover_c",
           "rover_cg", "oracle_cand", "oracle_comp", "control", "anti"]
WN_METHODS = ["first", "conf", "mbr", "cm05", "rover_freq", "rover_c", "rover_cg", "oracle_cand"]
CELL = ["model", "precision", "set", "kind", "arm", "k"]
WITHIN = [("conf", "rover_cg"), ("mbr", "rover_cg"), ("cm05", "rover_cg"), ("conf", "rover_c"),
          ("conf", "rover_freq"), ("first", "rover_cg"), ("conf", "mbr"), ("first", "conf"),
          ("rover_freq", "rover_cg")]
N_BOOT = 2000


# ---------------------------------------------------------------------------------- stats

def boot_delta(ea, eb, words, clusters=None, n_boot=N_BOOT, seed=0) -> dict:
    """Paired bootstrap of corpus WER(a) - WER(b), over utterances and over clusters."""
    ea, eb, w = (np.asarray(x, dtype=np.float64) for x in (ea, eb, words))
    n = len(w)
    if n == 0 or w.sum() == 0:
        return {}
    obs = 100.0 * (ea.sum() - eb.sum()) / w.sum()
    rng = np.random.default_rng(seed)
    d = np.empty(n_boot)
    step = max(1, min(n_boot, 4_000_000 // max(n, 1)))
    for s in range(0, n_boot, step):
        m = min(step, n_boot - s)
        idx = rng.integers(0, n, size=(m, n))
        d[s:s + m] = 100.0 * (ea[idx].sum(1) - eb[idx].sum(1)) / w[idx].sum(1)
    out = dict(delta=obs, ci_low=float(np.percentile(d, 2.5)), ci_high=float(np.percentile(d, 97.5)),
               se=float(d.std(ddof=1)), p_boot=float((d <= 0).mean()),
               n=n, n_better=int((eb < ea).sum()), n_worse=int((eb > ea).sum()), n_tied=int((ea == eb).sum()))
    if clusters is not None:
        cdf = pd.DataFrame({"c": clusters, "a": ea, "b": eb, "w": w}).groupby("c").sum()
        nc = len(cdf)
        out["n_clusters"] = nc
        if 5 <= nc < n:
            ca, cb, cw = cdf["a"].to_numpy(), cdf["b"].to_numpy(), cdf["w"].to_numpy()
            dc = np.empty(n_boot)
            step = max(1, min(n_boot, 4_000_000 // nc))
            for s in range(0, n_boot, step):
                m = min(step, n_boot - s)
                idx = rng.integers(0, nc, size=(m, nc))
                dc[s:s + m] = 100.0 * (ca[idx].sum(1) - cb[idx].sum(1)) / np.maximum(cw[idx].sum(1), 1)
            out.update(cl_ci_low=float(np.percentile(dc, 2.5)), cl_ci_high=float(np.percentile(dc, 97.5)))
    try:
        from scipy.stats import wilcoxon
        diff = ea - eb
        out["p_wilcoxon"] = float(wilcoxon(diff).pvalue) if np.any(diff != 0) else 1.0
    except Exception:
        out["p_wilcoxon"] = None
    return out


def random_effects(deltas: list[float], ses: list[float]) -> dict:
    """DerSimonian-Laird pooling of per-set deltas."""
    d, s = np.asarray(deltas, float), np.asarray(ses, float)
    ok = s > 0
    d, s = d[ok], s[ok]
    k = len(d)
    if k == 0:
        return {}
    w = 1.0 / s ** 2
    fixed = float((w * d).sum() / w.sum())
    q = float((w * (d - fixed) ** 2).sum())
    tau2 = max(0.0, (q - (k - 1)) / (w.sum() - (w ** 2).sum() / w.sum())) if k > 1 else 0.0
    wr = 1.0 / (s ** 2 + tau2)
    est = float((wr * d).sum() / wr.sum())
    se = float(math.sqrt(1.0 / wr.sum()))
    i2 = max(0.0, (q - (k - 1)) / q) if q > 0 and k > 1 else 0.0
    return dict(k_sets=k, pooled=est, ci_low=est - 1.96 * se, ci_high=est + 1.96 * se, tau2=tau2, I2=i2,
                n_positive=int((d > 0).sum()))


# ---------------------------------------------------------------------------------- tables

def load_tables(root: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    per, grid = [], []
    for p in sorted((root / "analysis").glob("*.parquet")):
        try:
            df = pd.read_parquet(p)
        except Exception as e:
            print(f"[aggregate] unreadable {p.name}: {e}", flush=True)
            continue
        (grid if p.name.endswith(".grid.parquet") else per).append(df)
    per_df = pd.concat(per, ignore_index=True) if per else pd.DataFrame()
    grid_df = pd.concat(grid, ignore_index=True) if grid else pd.DataFrame()
    if not per_df.empty:
        per_df["precision"] = per_df["precision"].fillna("")
        per_df = per_df.drop_duplicates(subset=CELL + ["id"], keep="last")
    if not grid_df.empty:
        grid_df["precision"] = grid_df["precision"].fillna("")
    return per_df, grid_df


def cells_table(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for key, g in df.groupby(CELL, sort=True):
        words = g["ref_len"].sum()
        r = dict(zip(CELL, key), n=len(g), words=int(words), audio_min=round(g["duration_s"].sum() / 60, 2),
                 n_clusters=g["cluster"].nunique(), mean_unique=g["n_unique"].mean(),
                 mean_pairwise=g["pairwise"].mean(), conf_frac=g["conf_frac"].mean(),
                 lang=g["lang"].iloc[0])
        for m in METHODS:
            col = f"e_{m}"
            if col in g and g[col].notna().all():
                r[f"wer_{m}"] = 100.0 * g[col].sum() / max(words, 1)
                r[f"mu_{m}"] = 100.0 * (g[col] / g["ref_len"].clip(lower=1)).mean()
        if "ref_len_wn" in g and g["ref_len_wn"].notna().all():
            wwn = g["ref_len_wn"].sum()
            for m in WN_METHODS:
                col = f"wn_{m}"
                if col in g and g[col].notna().all():
                    r[f"wn_wer_{m}"] = 100.0 * g[col].sum() / max(wwn, 1)
        if "e_oracle_cand" in g and g["e_oracle_comp"].notna().all():
            r["beats_best_pct"] = 100.0 * (g["e_oracle_comp"] < g["e_oracle_cand"]).mean()
        rows.append(r)
    return pd.DataFrame(rows)


def main_k(df_cell: pd.DataFrame) -> int:
    return int(df_cell["k"].max())


def contrasts_table(df: pd.DataFrame, k_extra=(16,)) -> pd.DataFrame:
    """Paired contrasts for the arms the paper reports. Sweeps and ablations are summarised by
    cells.csv; running every bootstrap for them too costs minutes and says nothing new."""
    rows = []
    df = df[df["kind"].isin(["main", "kscale", "lowT"])]
    for key, g in df.groupby(["model", "precision", "set", "kind", "arm"], sort=True):
        kmax = int(g["k"].max())
        for k in sorted({kmax, *[x for x in k_extra if x < kmax and (g["k"] == x).any()]}):
            gk = g[g["k"] == k]
            for a, b in WITHIN:
                for pre, words_col in (("e_", "ref_len"), ("wn_", "ref_len_wn")):
                    ca, cb = f"{pre}{a}", f"{pre}{b}"
                    if ca not in gk or cb not in gk or gk[ca].isna().any() or gk[cb].isna().any():
                        continue
                    if gk[words_col].isna().any():
                        continue
                    s = boot_delta(gk[ca], gk[cb], gk[words_col], gk["cluster"].to_numpy())
                    if s:
                        rows.append(dict(zip(["model", "precision", "set", "kind", "arm"], key), k=k,
                                         k_is_main=(k == kmax), norm="legacy" if pre == "e_" else "whisper",
                                         a=a, b=b, **s))
    return pd.DataFrame(rows)


def _paired(df_a: pd.DataFrame, col_a: str, df_b: pd.DataFrame, col_b: str):
    m = df_a[["id", "cluster", "ref_len", col_a]].merge(df_b[["id", col_b]], on="id", suffixes=("_a", "_b"))
    ca = col_a + "_a" if col_a == col_b else col_a
    cb = col_b + "_b" if col_a == col_b else col_b
    return m, ca, cb


def cross_arm_table(df: pd.DataFrame, cfg: dict) -> pd.DataFrame:
    """Composition from a model's candidate pool against that model's standard decoding."""
    rows = []
    # (model, (kind, arm, k, method) for a, the same for b, label); a and b may live in
    # different job kinds, e.g. Drax's standard decode against its low-temperature pool
    specs = []
    kd = cfg["k_main"]["drax"]
    kw = cfg["k_main"]["whisfusion"]
    lo = cfg.get("drax_low_T", 0.4)
    ref_K = cfg["drax"]["ref_K"]
    specs += [("drax", ("main", "ref", 1, "first"), ("main", "main", kd, "rover_cg"), "Drax standard (T=0.1, 1 sample) vs ROVER@main"),
              ("drax", ("main", "ref", 1, "first"), ("main", "main", kd, "conf"), "Drax standard vs confidence pick@main"),
              ("drax", ("main", "ref", 1, "first"), ("main", "main", kd, "mbr"), "Drax standard vs MBR pick@main"),
              ("drax", ("main", "ref", ref_K, "rover_cg"), ("main", "main", kd, "rover_cg"), "ROVER@T=0.1 vs ROVER@main"),
              ("drax", ("main", "ref", 1, "first"), ("lowT", "lowT", kd, "rover_cg"), f"Drax standard vs ROVER@T={lo:g}"),
              ("drax", ("main", "ref", 1, "first"), ("lowT", "lowT", kd, "mbr"), f"Drax standard vs MBR@T={lo:g}"),
              ("drax", ("lowT", "lowT", kd, "mbr"), ("lowT", "lowT", kd, "rover_cg"), f"MBR vs ROVER, both @T={lo:g}"),
              ("drax", ("main", "main", kd, "rover_cg"), ("lowT", "lowT", kd, "rover_cg"), f"ROVER@main vs ROVER@T={lo:g}"),
              ("whisfusion", ("main", "main", 15, "conf"), ("main", "main", kw, "rover_cg"),
               "Whisfusion upstream (K=15 confidence) vs ROVER@main"),
              ("whisfusion", ("main", "main", 15, "conf"), ("main", "main", 15, "rover_cg"),
               "Whisfusion upstream vs ROVER at the same K=15")]
    for m in ("whisper-small", "whisper-turbo"):
        K = cfg["whisper_sample"]["K"]
        B = cfg["whisper_beams"]
        specs += [(m, ("main", "greedy", 1, "first"), ("main", "sample", K, "rover_cg"), "greedy vs ROVER over samples"),
                  (m, ("main", "greedy", 1, "first"), ("main", "beam", B, "rover_cg"), "greedy vs ROVER over beam n-best"),
                  (m, ("main", "beam", 1, "first"), ("main", "beam", B, "rover_cg"), "top beam vs ROVER over n-best"),
                  (m, ("main", "greedy", 1, "first"), ("main", "sample", K, "mbr"), "greedy vs MBR over samples")]
    specs += [("parakeet-ctc", ("main", "greedy", 1, "first"),
               ("main", "sample", cfg["ctc_sample"]["K"], "rover_cg"), "greedy vs ROVER over sampled CTC paths")]

    for model, (kind_a, arm_a, k_a, m_a), (kind_b, arm_b, k_b, m_b), label in specs:
        base = df[(df["model"] == model) & (df["precision"] == "")]
        for s, g in base.groupby("set"):
            ga = g[(g["kind"] == kind_a) & (g["arm"] == arm_a) & (g["k"] == k_a)]
            gb = g[(g["kind"] == kind_b) & (g["arm"] == arm_b) & (g["k"] == k_b)]
            if ga.empty or gb.empty:
                continue
            m, ca, cb = _paired(ga, f"e_{m_a}", gb, f"e_{m_b}")
            if m.empty:
                continue
            st = boot_delta(m[ca], m[cb], m["ref_len"], m["cluster"].to_numpy())
            if st:
                rows.append(dict(model=model, set=s, label=label, a=f"{m_a}@{arm_a}/k{k_a}",
                                 b=f"{m_b}@{arm_b}/k{k_b}",
                                 wer_a=100.0 * m[ca].sum() / m["ref_len"].sum(),
                                 wer_b=100.0 * m[cb].sum() / m["ref_len"].sum(), **st))
    return pd.DataFrame(rows)


def tuned_table(df: pd.DataFrame, grid: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
    """Pick lambda and (alpha, eps, gamma) on dev, per model; report tuned systems on test."""
    if grid.empty:
        return pd.DataFrame(), {}
    rows, chosen = [], {}
    main_arms = {"drax": "main", "whisfusion": "main", "whisper-small": "sample",
                 "whisper-turbo": "sample", "parakeet-ctc": "sample"}
    for model, arm in main_arms.items():
        g = grid[(grid["model"] == model) & (grid["arm"] == arm) & (grid["kind"] == "main") &
                 (grid["precision"] == "")]
        if g.empty:
            continue
        dev = g[g["set"].isin(C.DEV_SETS)]
        if dev.empty:
            continue
        rv = dev[dev["param"] == "rover"]
        sl = dev[dev["param"] == "select"]
        w = dev.drop_duplicates(["set", "id"])["ref_len"].sum()
        rv_w = rv.groupby(["alpha", "eps", "gamma"])["e"].sum() / max(w, 1) * 100
        sl_w = sl.groupby("lam")["e"].sum() / max(w, 1) * 100
        (a, e, gm), lam = rv_w.idxmin(), sl_w.idxmin()
        chosen[model] = dict(alpha=a, eps=e, gamma=gm, lam=lam, dev_rover=float(rv_w.min()),
                             dev_select=float(sl_w.min()), n_dev=int(len(dev.drop_duplicates(["set", "id"]))))
        test = g[~g["set"].isin(C.DEV_SETS)]
        for s, gs in test.groupby("set"):
            r = gs[(gs["param"] == "rover") & (gs["alpha"] == a) & (gs["eps"] == e) &
                   (gs["gamma"] == gm)].set_index("id")
            q = gs[(gs["param"] == "select") & (gs["lam"] == lam)].set_index("id")
            ids = r.index.intersection(q.index)
            if len(ids) == 0:
                continue
            r, q = r.loc[ids], q.loc[ids]
            st = boot_delta(q["e"], r["e"], q["ref_len"], q["cluster"].to_numpy())
            rows.append(dict(model=model, set=s, k=int(gs["k"].iloc[0]), alpha=a, eps=e, gamma=gm, lam=lam,
                             wer_select_tuned=100 * q["e"].sum() / q["ref_len"].sum(),
                             wer_rover_tuned=100 * r["e"].sum() / q["ref_len"].sum(), **st))
    return pd.DataFrame(rows), chosen


def timing_table(root: Path, df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    plan = {j["id"]: j for j in json.load(open(root / "plan.json", encoding="utf-8"))}
    stats = []
    for p in (root / "state" / "done").glob("*.json"):
        s = json.load(open(p, encoding="utf-8"))
        j = plan.get(p.stem)
        if not j or "decode_s" not in s:
            continue
        stats.append((j, s))
    by: dict[tuple, dict] = {}
    for j, s in stats:
        for a in j["arms"]:
            key = (j["model"], j.get("precision") or "", j["kind"], a["name"])
            d = by.setdefault(key, dict(n=0, audio_s=0.0, encode_s=0.0, decode_s=0.0, wall_s=0.0, peak_mb=0.0,
                                        K=a.get("K", a.get("beams", 1)), T=a.get("T"), steps=a.get("steps")))
            d["n"] += s.get("n_ok", 0)
            d["audio_s"] += s.get("audio_s", 0.0)
            d["encode_s"] += s.get("encode_s", 0.0) / max(len(j["arms"]), 1)
            d["decode_s"] += s["decode_s"].get(a["name"], 0.0)
            d["peak_mb"] = max(d["peak_mb"], s.get("peak_mem_mb", 0.0) or 0.0)
    for (model, prec, kind, arm), d in by.items():
        if d["n"] == 0:
            continue
        rms = df[(df["model"] == model) & (df["arm"] == arm) & (df["kind"] == kind)]
        rms = rms[rms["k"] == rms["k"].max()]["rover_ms"].mean() if not rms.empty else None
        rows.append(dict(model=model, precision=prec, kind=kind, arm=arm, K=d["K"], T=d["T"], steps=d["steps"],
                         n=d["n"], audio_h=d["audio_s"] / 3600, s_per_utt=(d["encode_s"] + d["decode_s"]) / d["n"],
                         encode_s_per_utt=d["encode_s"] / d["n"], decode_s_per_utt=d["decode_s"] / d["n"],
                         rtf=(d["encode_s"] + d["decode_s"]) / max(d["audio_s"], 1e-9),
                         rtf_decode=d["decode_s"] / max(d["audio_s"], 1e-9),
                         ms_per_candidate=1000 * d["decode_s"] / d["n"] / max(d["K"] or 1, 1),
                         rover_ms_per_utt=rms, peak_mem_mb=d["peak_mb"]))
    return pd.DataFrame(rows)


def meta_table(contr: pd.DataFrame) -> pd.DataFrame:
    rows = []
    if contr.empty:
        return pd.DataFrame()
    c = contr[(contr["k_is_main"]) & (contr["kind"] == "main") & (~contr["set"].isin(C.DEV_SETS))]
    for (model, prec, arm, norm, a, b), g in c.groupby(["model", "precision", "arm", "norm", "a", "b"]):
        re_ = random_effects(g["delta"].tolist(), g["se"].tolist())
        if re_:
            sig = int(((g["ci_low"] > 0)).sum())
            rows.append(dict(model=model, precision=prec, arm=arm, norm=norm, a=a, b=b, n_sig_positive=sig,
                             n_sig_negative=int((g["ci_high"] < 0).sum()), **re_))
    return pd.DataFrame(rows)


# ---------------------------------------------------------------------------------- report

def fmt(x, nd=2):
    return "" if x is None or (isinstance(x, float) and math.isnan(x)) else f"{x:.{nd}f}"


def write_report(out: Path, cells: pd.DataFrame, contr: pd.DataFrame, cross: pd.DataFrame,
                 tuned: pd.DataFrame, chosen: dict, meta: pd.DataFrame, timing: pd.DataFrame,
                 summary: dict) -> None:
    L = [f"# Campaign report\n", f"generated {time.strftime('%Y-%m-%d %H:%M UTC', time.gmtime())}\n",
         f"utterance decodes analysed: {summary.get('n_rows', 0)}; cells: {len(cells)}\n"]
    if not cells.empty:
        L.append("\n## Main arm at the largest K, corpus WER (legacy normalisation)\n")
        L.append("| model | set | n | K | first | conf | MBR | ROVER | ROVER-cg | oracle | oracle-comp | control | pairwise % |")
        L.append("|---|---|---|---|---|---|---|---|---|---|---|---|---|")
        main = cells[(cells["kind"] == "main") & (cells["precision"] == "")]
        for (model, s, arm), g in main.groupby(["model", "set", "arm"]):
            r = g[g["k"] == g["k"].max()].iloc[0]
            L.append(f"| {model}/{arm} | {s} | {r['n']} | {r['k']} | {fmt(r.get('wer_first'))} | "
                     f"{fmt(r.get('wer_conf'))} | {fmt(r.get('wer_mbr'))} | {fmt(r.get('wer_rover_c'))} | "
                     f"{fmt(r.get('wer_rover_cg'))} | {fmt(r.get('wer_oracle_cand'))} | "
                     f"{fmt(r.get('wer_oracle_comp'))} | {fmt(r.get('wer_control'))} | {fmt(r.get('mean_pairwise'), 1)} |")
    if not contr.empty:
        L.append("\n## ROVER-cg against the best selection, paired bootstrap (main arm, largest K)\n")
        L.append("| model | set | vs | delta | 95% CI | cluster CI | Wilcoxon p | better/worse/tied |")
        L.append("|---|---|---|---|---|---|---|---|")
        c = contr[(contr["k_is_main"]) & (contr["norm"] == "legacy") & (contr["kind"] == "main") &
                  (contr["b"] == "rover_cg") & (contr["a"].isin(["conf", "mbr"]))]
        for _, r in c.iterrows():
            L.append(f"| {r['model']}/{r['arm']} | {r['set']} | {r['a']} | {r['delta']:+.2f} | "
                     f"[{r['ci_low']:+.2f}, {r['ci_high']:+.2f}] | "
                     f"[{fmt(r.get('cl_ci_low'))}, {fmt(r.get('cl_ci_high'))}] | "
                     f"{fmt(r.get('p_wilcoxon'), 4)} | {r['n_better']}/{r['n_worse']}/{r['n_tied']} |")
    if not meta.empty:
        L.append("\n## Pooled over test sets (random effects)\n")
        L.append("| model | arm | norm | a -> b | sets | pooled delta | 95% CI | I2 | sets sig + / - |")
        L.append("|---|---|---|---|---|---|---|---|---|")
        for _, r in meta.iterrows():
            L.append(f"| {r['model']} | {r['arm']} | {r['norm']} | {r['a']} -> {r['b']} | {r['k_sets']} | "
                     f"{r['pooled']:+.2f} | [{r['ci_low']:+.2f}, {r['ci_high']:+.2f}] | {r['I2']:.2f} | "
                     f"{r['n_sig_positive']} / {r['n_sig_negative']} |")
    if not cross.empty:
        L.append("\n## Against each model's standard decoding\n")
        L.append("| model | set | comparison | WER a | WER b | delta | 95% CI |")
        L.append("|---|---|---|---|---|---|---|")
        for _, r in cross.iterrows():
            L.append(f"| {r['model']} | {r['set']} | {r['label']} | {r['wer_a']:.2f} | {r['wer_b']:.2f} | "
                     f"{r['delta']:+.2f} | [{r['ci_low']:+.2f}, {r['ci_high']:+.2f}] |")
    if chosen:
        L.append("\n## Tuned on dev (ls-dev-clean + ls-dev-other), reported on test\n")
        for m, c in chosen.items():
            L.append(f"- **{m}**: alpha={c['alpha']}, eps={c['eps']}, gamma={c['gamma']}, lambda={c['lam']} "
                     f"(dev: select {c['dev_select']:.2f}, ROVER {c['dev_rover']:.2f}, n={c['n_dev']})")
        if not tuned.empty:
            L.append("\n| model | set | tuned select | tuned ROVER | delta | 95% CI |")
            L.append("|---|---|---|---|---|---|")
            for _, r in tuned.iterrows():
                L.append(f"| {r['model']} | {r['set']} | {r['wer_select_tuned']:.2f} | {r['wer_rover_tuned']:.2f} | "
                         f"{r['delta']:+.2f} | [{r['ci_low']:+.2f}, {r['ci_high']:+.2f}] |")
    if not timing.empty:
        L.append("\n## Cost\n")
        L.append("| model | kind | arm | K | T | steps | n | s/utt | encode s | decode s | RTF | ms/candidate | ROVER ms | peak MB |")
        L.append("|---|---|---|---|---|---|---|---|---|---|---|---|---|---|")
        for _, r in timing.sort_values(["model", "kind", "arm"]).iterrows():
            L.append(f"| {r['model']}{'@' + r['precision'] if r['precision'] else ''} | {r['kind']} | {r['arm']} | "
                     f"{r['K']} | {fmt(r['T'], 2)} | {r['steps'] or ''} | {r['n']} | {r['s_per_utt']:.3f} | "
                     f"{r['encode_s_per_utt']:.3f} | {r['decode_s_per_utt']:.3f} | {r['rtf']:.4f} | "
                     f"{r['ms_per_candidate']:.1f} | {fmt(r['rover_ms_per_utt'])} | {r['peak_mem_mb']:.0f} |")
    (out / "REPORT.md").write_text("\n".join(L) + "\n", encoding="utf-8")


def main() -> int:
    ap = argparse.ArgumentParser()
    ap.add_argument("--root", required=True)
    ap.add_argument("--no_csv", action="store_true")
    args = ap.parse_args()
    root = Path(args.root)
    out = root / "results"
    out.mkdir(parents=True, exist_ok=True)
    cfg = json.load(open(root / "run_config.json", encoding="utf-8"))
    t0 = time.time()

    df, grid = load_tables(root)
    summary: dict = {"generated_unix": time.time(), "n_rows": int(len(df)), "config": cfg}
    if df.empty:
        print("[aggregate] nothing to aggregate", flush=True)
        (out / "campaign_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
        return 0
    df.to_parquet(out / "per_utt_k.parquet", index=False)
    if not grid.empty:
        grid.to_parquet(out / "grid.parquet", index=False)
    if not args.no_csv:
        df.to_csv(out / "per_utt_k.csv.gz", index=False, compression="gzip")

    cells = cells_table(df)
    cells.to_csv(out / "cells.csv", index=False)
    print(f"[aggregate] cells {len(cells)} ({time.time() - t0:.0f} s)", flush=True)
    contr = contrasts_table(df)
    contr.to_csv(out / "contrasts.csv", index=False)
    print(f"[aggregate] contrasts {len(contr)} ({time.time() - t0:.0f} s)", flush=True)
    cross = cross_arm_table(df, cfg)
    cross.to_csv(out / "cross_arm.csv", index=False)
    tuned, chosen = tuned_table(df, grid)
    tuned.to_csv(out / "tuned.csv", index=False)
    meta = meta_table(contr)
    meta.to_csv(out / "meta_analysis.csv", index=False)
    timing = timing_table(root, df)
    timing.to_csv(out / "timing.csv", index=False)

    datasets = {}
    for p in sorted((root / "manifests").glob("*.meta.json")):
        datasets[p.name.split(".meta")[0]] = json.load(open(p, encoding="utf-8"))
    summary.update(
        datasets=datasets,
        models={m: C.MODELS[m] for m in df["model"].unique() if m in C.MODELS},
        tuned_on_dev=chosen,
        cells=cells.replace({np.nan: None}).to_dict(orient="records"),
        contrasts_main=contr[contr["k_is_main"]].replace({np.nan: None}).to_dict(orient="records") if not contr.empty else [],
        cross_arm=cross.replace({np.nan: None}).to_dict(orient="records"),
        tuned=tuned.replace({np.nan: None}).to_dict(orient="records"),
        meta_analysis=meta.replace({np.nan: None}).to_dict(orient="records"),
        timing=timing.replace({np.nan: None}).to_dict(orient="records"),
    )
    with open(out / "campaign_summary.json", "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=1, default=lambda o: o.item() if hasattr(o, "item") else str(o))
    write_report(out, cells, contr, cross, tuned, chosen, meta, timing, summary)
    print(f"[aggregate] done in {time.time() - t0:.0f} s -> {out}", flush=True)
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


In [ ]:
%%writefile /kaggle/working/code/src/campaign/run.py
#!/usr/bin/env python3
"""Orchestrator: data prep, one lane per GPU, analysis in the background, aggregation at the end.

    python -m campaign.run --config run_config.json --root /kaggle/working/campaign --data /tmp/cdata

A lane is a thread that keeps one GPU busy. It asks the shared state which model has the
lowest-tier work ready, starts a worker process for that model pinned to its GPU
(CUDA_VISIBLE_DEVICES), and when the worker exits (its model ran out of work, or it yielded to
lower-tier work elsewhere) picks again. Two independent processes rather than DDP: the models
are small enough that data parallelism across utterances is all that is needed.

Deadlines: GPU work stops `tail_minutes` before `run_hours`; the tail finishes the analysis of
every dump and writes the results. Everything written before a crash or a timeout is kept.
"""

from __future__ import annotations

import argparse
import concurrent.futures as cf
import json
import os
import platform
import shutil
import subprocess
import sys
import threading
import time
import traceback
from pathlib import Path

HERE = Path(__file__).resolve().parent
SRC = HERE.parent
sys.path.insert(0, str(SRC))

from campaign import config as C  # noqa: E402
from campaign import state as ST  # noqa: E402

T_START = time.time()


def say(msg: str) -> None:
    el = (time.time() - T_START) / 60
    print(f"[{time.strftime('%H:%M:%S')} +{el:6.1f}m] {msg}", flush=True)


def env_info() -> dict:
    info = dict(python=platform.python_version(), host=platform.node())
    try:
        import torch
        info.update(torch=torch.__version__, cuda=torch.version.cuda, n_gpus=torch.cuda.device_count(),
                    gpus=[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
    except Exception as e:
        info["torch"] = f"ERROR {e}"
    for mod in ("transformers", "huggingface_hub", "numpy", "pandas", "rapidfuzz", "soundfile", "librosa"):
        try:
            info[mod] = str(__import__(mod).__version__)
        except Exception as e:
            info[mod] = f"ERROR {type(e).__name__}"
    try:
        info["nvidia_smi"] = subprocess.run(
            ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
            capture_output=True, text=True, timeout=10).stdout.strip()
    except Exception:
        pass
    try:
        du = shutil.disk_usage("/tmp")
        info["tmp_free_gb"] = round(du.free / 1e9, 1)
        info["cpu_count"] = os.cpu_count()
    except Exception:
        pass
    return info


class Lane(threading.Thread):
    def __init__(self, gpu: int, st: ST.State, args, preferred: list[str], gpu_deadline: float,
                 busy: dict, lock: threading.Lock):
        super().__init__(daemon=True)
        self.gpu, self.st, self.args = gpu, st, args
        self.preferred = preferred
        self.deadline = gpu_deadline
        self.busy, self.lock = busy, lock
        self.current: str | None = None
        self.proc: subprocess.Popen | None = None
        self.launches = 0
        self.idle_s = 0.0

    def run(self):
        while time.time() < self.deadline - 60:
            with self.lock:
                others = {m for g, m in self.busy.items() if g != self.gpu and m}
                model = self.st.choose_model(self.current, self.preferred, others)
                if model:
                    self.busy[self.gpu] = model
            if model is None:
                if self.st.any_waiting() or not self.st.prep_finished():
                    time.sleep(15)
                    self.idle_s += 15
                    continue
                say(f"lane {self.gpu}: no work left")
                break
            self.current = model
            self.launches += 1
            say(f"lane {self.gpu}: starting {model}")
            env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(self.gpu), PYTHONUNBUFFERED="1")
            log = self.st.logs / f"worker_gpu{self.gpu}_{self.launches:03d}_{model.replace('@', '_')}.log"
            cmd = [sys.executable, "-m", "campaign.worker", "--model", model, "--lane", str(self.gpu),
                   "--root", str(self.st.root), "--data", str(self.st.data), "--deadline", str(self.deadline)]
            with open(log, "w", encoding="utf-8") as fh:
                self.proc = subprocess.Popen(cmd, env=env, stdout=fh, stderr=subprocess.STDOUT)
                rc = self.proc.wait()
            with self.lock:
                self.busy[self.gpu] = None
            for jid in self.st.orphaned_claims(self.gpu):
                say(f"lane {self.gpu}: releasing orphaned claim {jid} (worker rc={rc})")
                self.st.release(jid, crashed=True)
            text = log.read_text(encoding="utf-8", errors="replace")
            lines = text.splitlines()
            errs = [ln for ln in lines if "[!]" in ln or "Traceback" in ln or "CRASHED" in ln]
            dones = [ln.split("] ", 2)[-1] for ln in lines if "] done " in ln]
            say(f"lane {self.gpu}: {model} exited rc={rc}, {len(dones)} jobs, {len(errs)} error lines")
            for ln in dones[-6:]:
                say(f"    {ln}")
            for ln in errs[:4]:
                say(f"    ERR {ln[-400:]}")
            if rc != 0:
                say(f"lane {self.gpu}: worker {model} log tail:\n{text[-3000:]}")
                if rc not in (0, 3):
                    time.sleep(5)
        say(f"lane {self.gpu}: finished ({self.launches} worker launches, idle {self.idle_s / 60:.1f} min)")


def analysis_task(root: str, job: dict, k_main: int) -> list[dict]:
    sys.path.insert(0, str(SRC))
    from campaign import analysis
    try:
        os.nice(5)
    except Exception:
        pass
    return analysis.analyze_job(Path(root), job, k_main)


def k_main_of(cfg: dict, job: dict) -> int:
    return cfg["k_main"].get(job["model"], 64)


def progress(st: ST.State, cfg: dict) -> dict:
    counts = st.counts()
    by_model: dict[str, dict] = {}
    for p in st.done_dir.glob("*.json"):
        try:
            s = json.load(open(p, encoding="utf-8"))
        except Exception:
            continue
        m = s.get("model", p.stem.split("__")[0])
        d = by_model.setdefault(m, dict(jobs=0, utts=0, audio_h=0.0, wall_h=0.0))
        d["jobs"] += 1
        d["utts"] += s.get("n_ok", 0)
        d["audio_h"] += s.get("audio_s", 0.0) / 3600
        d["wall_h"] += s.get("wall_s", 0.0) / 3600
    tiers: dict[int, dict] = {}
    for j in st.plan():
        t = tiers.setdefault(j["tier"], {"total": 0, "done": 0})
        t["total"] += 1
        t["done"] += st.is_done(j["id"])
    return dict(elapsed_h=(time.time() - T_START) / 3600, jobs=counts, by_model=by_model, tiers=tiers)


def main() -> int:
    ap = argparse.ArgumentParser()
    ap.add_argument("--config", required=True)
    ap.add_argument("--root", required=True)
    ap.add_argument("--data", required=True)
    ap.add_argument("--n_gpus", type=int, default=None)
    args = ap.parse_args()

    cfg = C.load_config(args.config)
    root, data = Path(args.root), Path(args.data)
    root.mkdir(parents=True, exist_ok=True)
    t_end = T_START + cfg["run_hours"] * 3600
    gpu_deadline = t_end - cfg["tail_minutes"] * 60
    cfg["t_start_unix"], cfg["t_end_unix"], cfg["gpu_deadline_unix"] = T_START, t_end, gpu_deadline
    C.write_json(root / "run_config.json", cfg)

    _plans = {"round2": C.build_plan_round2, "sweep": C.build_plan_sweep,
              "tree": C.build_plan_tree}
    plan = _plans.get(cfg.get("plan"), C.build_plan)(cfg)
    C.write_json(root / "plan.json", plan)
    info = env_info()
    C.write_json(root / "env.json", info)
    say(f"env: {json.dumps(info)}")
    tiers = {}
    for j in plan:
        tiers[j["tier"]] = tiers.get(j["tier"], 0) + 1
    say(f"plan: {len(plan)} jobs, per tier {dict(sorted(tiers.items()))}; GPU work until "
        f"+{(gpu_deadline - T_START) / 3600:.2f} h, results by +{(t_end - T_START) / 3600:.2f} h")

    st = ST.State(root, data)

    # data prep in the background; manifests appear one by one
    prep_log = open(st.logs / "prep.log", "a", encoding="utf-8")
    prep = subprocess.Popen([sys.executable, "-m", "campaign.prep_data", "--config", str(root / "run_config.json"),
                             "--data", str(data)], stdout=prep_log, stderr=subprocess.STDOUT,
                            env=dict(os.environ, PYTHONUNBUFFERED="1"))

    n_gpus = args.n_gpus if args.n_gpus is not None else max(1, info.get("n_gpus", 0) or 0)
    if not info.get("n_gpus"):
        say("no CUDA device: running one CPU lane (debug mode)")
    lock = threading.Lock()
    busy: dict = {}
    prefs = [["drax", "whisper-turbo", "whisfusion", "whisper-small", "parakeet-ctc"],
             ["whisfusion", "whisper-small", "parakeet-ctc", "drax", "whisper-turbo"]]
    lanes = [Lane(g, st, args, prefs[g % 2], gpu_deadline, busy, lock) for g in range(n_gpus)]
    for ln in lanes:
        ln.start()

    pool = cf.ProcessPoolExecutor(max_workers=max(1, cfg["analysis_workers"]))
    submitted: dict[str, cf.Future] = {}
    plan_by_id = {j["id"]: j for j in plan}
    last_prog = 0.0

    def submit_finished():
        for p in st.done_dir.glob("*.json"):
            jid = p.stem
            if jid in submitted or jid not in plan_by_id:
                continue
            job = plan_by_id[jid]
            if all((st.analysis / f"{jid}__{a['name']}.parquet").exists() for a in job["arms"]):
                submitted[jid] = None
                continue
            submitted[jid] = pool.submit(analysis_task, str(root), job, k_main_of(cfg, job))

    prep_reported = False
    while any(ln.is_alive() for ln in lanes):
        time.sleep(20)
        submit_finished()
        if prep.poll() is not None and not prep_reported:
            prep_reported = True
            say(f"data prep finished rc={prep.returncode}; "
                f"failed sets: {[p.stem for p in st.manifests.glob('*.failed')]}")
        if time.time() - last_prog > 600:
            last_prog = time.time()
            pr = progress(st, cfg)
            C.write_json(root / "progress.json", pr)
            done_an = sum(1 for f in submitted.values() if f is not None and f.done())
            say(f"progress: jobs {pr['jobs']}; tiers "
                f"{ {t: f'{v['done']}/{v['total']}' for t, v in sorted(pr['tiers'].items())} }; "
                f"analysed {done_an}/{len(submitted)}")
            for m, d in sorted(pr["by_model"].items()):
                say(f"    {m:<16} {d['jobs']:>4} jobs {d['utts']:>6} utts {d['audio_h']:6.2f} h audio "
                    f"{d['wall_h']:6.2f} GPU-h")
        if time.time() > t_end - 5 * 60:
            say("hard end approaching; abandoning lanes")
            for ln in lanes:
                if ln.proc is not None and ln.proc.poll() is None:
                    ln.proc.kill()
            break

    if prep.poll() is None:
        prep.terminate()
    say("GPU phase over; finishing analysis")
    submit_finished()
    for jid, fut in list(submitted.items()):
        if fut is None:
            continue
        remaining = t_end - 8 * 60 - time.time()
        if remaining <= 0:
            say("out of time for analysis; the rest can be run offline with campaign.analysis")
            break
        try:
            fut.result(timeout=remaining)
        except Exception as e:
            say(f"analysis of {jid} failed: {type(e).__name__}: {e}")
    pool.shutdown(wait=False, cancel_futures=True)

    # provenance: exactly which utterances every set held
    (root / "manifests").mkdir(exist_ok=True)
    for p in st.manifests.glob("*"):
        if p.suffix in (".jsonl", ".json") or p.name.endswith(".failed"):
            shutil.copy(p, root / "manifests" / p.name)

    pr = progress(st, cfg)
    C.write_json(root / "progress.json", pr)
    say("aggregating")
    try:
        remaining = max(60, t_end - time.time() - 60)
        rc = subprocess.run([sys.executable, "-m", "campaign.aggregate", "--root", str(root)],
                            timeout=remaining, env=dict(os.environ, PYTHONUNBUFFERED="1")).returncode
        say(f"aggregate rc={rc}")
    except subprocess.TimeoutExpired:
        say("aggregate timed out; run `python -m campaign.aggregate --root ...` offline")
    except Exception:
        traceback.print_exc()
    pack(root)
    say(f"done in {(time.time() - T_START) / 3600:.2f} h")
    return 0


def pack(root: Path) -> None:
    """Thousands of small files become a handful of tarballs; results/ stays browsable."""
    import tarfile
    for name in ("dumps", "analysis", "state", "logs", "manifests"):
        d = root / name
        if not d.exists():
            continue
        try:
            with tarfile.open(root / f"{name}.tar", "w") as tf:
                tf.add(d, arcname=name)
            shutil.rmtree(d)
            say(f"packed {name}.tar ({(root / f'{name}.tar').stat().st_size / 1e6:.0f} MB)")
        except Exception as e:
            say(f"packing {name} failed: {e}")


if __name__ == "__main__":
    raise SystemExit(main())


The decoder as committed, for the identity check further down. Flat sampling has to come out bit-identical to it, or every number measured before the tree work stops being comparable.

In [ ]:
%%writefile /kaggle/working/code/src/decode_old.py
"""Parallel Diffusion Decoding, returning all K candidates instead of the winner.

Step 0 masks everything and the update is argmax, so all K candidates are identical after
it; divergence comes from the masks of later steps. first_step_sampling samples there.
"""

from __future__ import annotations

from dataclasses import asdict, dataclass, field

import torch

DEFAULT_SCHEDULE = [1.0, 0.9, 0.85, 0.8]
WORD_START = "\u2581"  # SentencePiece word boundary


def _word_confidences(tokenizer, ids, confs, text) -> list[float] | None:
    """Mean confidence per normalised word, or None if it will not line up."""
    from scorers import normalize

    special = set(tokenizer.all_special_ids)
    pairs = [(i, c) for i, c in zip(ids, confs) if i not in special]
    if not pairs:
        return None
    keep_ids, keep_conf = zip(*pairs)
    pieces = tokenizer.convert_ids_to_tokens(list(keep_ids))

    words, buf, cur = [], [], []
    for piece, c in zip(pieces, keep_conf):
        if piece.startswith(WORD_START) and buf:
            words.append(("".join(buf), cur))
            buf, cur = [], []
        buf.append(piece.lstrip(WORD_START))
        cur.append(float(c))
    if buf:
        words.append(("".join(buf), cur))

    out = [sum(cs) / len(cs) for w, cs in words if normalize(w)]
    return out if len(out) == len(normalize(text).split()) else None


@dataclass
class Candidate:
    text: str
    avg_conf: float  # upstream selection metric: mean max-prob over non-pad positions
    min_conf: float
    median_conf: float
    mean_logprob: float
    mean_entropy: float
    n_tokens: int
    word_conf: list[float] | None = field(default=None)
    tokens: list[int] | None = field(default=None)


@dataclass
class DecodeResult:
    candidates: list[Candidate]
    n_unique: int
    identical_after_step1: bool
    n_candidates_used: int = 0      # differs from the request only when adaptive is on
    uncertainty: float | None = None  # mean (1 - max prob) at the probe step, if measured
    branch_widths: list[int] | None = None   # rows actually decoded at each step


@torch.no_grad()
def pdd_decode(
    wf,
    condition: torch.Tensor,
    n_candidates: int = 15,
    n_steps: int = 4,
    mask_ratio_schedule: list[float] | None = None,
    seq_len: int = 256,
    first_step_sampling: bool = False,
    temperature: float = 1.0,
    save_tokens: bool = False,
    seed: int | None = None,
    branch_schedule: list[int] | None = None,
    mask_mode: str = "uniform",
    adaptive: dict | None = None,
) -> DecodeResult:
    """branch_schedule gives the number of distinct mask groups at each step.

    Flat sampling is every candidate drawing its own mask at every step, which is
    branch_schedule = [K, K, K, K] and is what None means. A tree shares masks early and
    splits later: [1, 4, 32, 32] decodes one row, then four, then thirty-two, so siblings
    carry a common prefix. Rows are only materialised when they split, so a narrow early
    schedule is also cheaper, not just more correlated.

    mask_mode "uncertain" draws the mask with probability proportional to 1 - confidence
    from the previous step instead of uniformly, keeping the same expected mask ratio, so
    re-prediction is spent where the model is unsure rather than spread evenly.

    adaptive sizes the whole tree per utterance: after probe_step the width is set from how
    uncertain the shared prefix is, so an easy utterance gets a small tree and a hard one a
    large tree. dict(base, u0, gamma, k_min, k_max, probe_step); K = base * (u/u0) ** gamma.
    """
    schedule = mask_ratio_schedule or DEFAULT_SCHEDULE
    device = wf.device
    mask_id = wf.mask_token_id
    pad_id = wf.pad_token_id

    gen = None
    if seed is not None:
        gen = torch.Generator(device=device)
        gen.manual_seed(seed)

    bos = wf.tokenizer.bos_token_id
    bos = 0 if bos is None else bos

    widths = list(branch_schedule) if branch_schedule else [n_candidates] * n_steps
    widths = [max(1, int(w)) for w in widths[:n_steps]]
    widths += [widths[-1]] * (n_steps - len(widths))
    probe = int((adaptive or {}).get("probe_step", 1))

    rows = widths[0]
    cur = torch.full((rows, seq_len), mask_id, dtype=torch.long, device=device)
    cur[:, 0] = bos

    final_logits = None
    after_step1 = None
    conf_prev = None
    uncertainty = None
    used = []

    for step in range(n_steps):
        ratio = schedule[step] if step < len(schedule) else 0.7

        want = widths[step]
        if want > rows:                       # split: every parent takes the same children
            idx = torch.arange(want, device=device) % rows if want % rows else None
            cur = cur[idx] if idx is not None else cur.repeat_interleave(want // rows, dim=0)
            if conf_prev is not None:
                conf_prev = conf_prev[idx] if idx is not None else \
                    conf_prev.repeat_interleave(want // rows, dim=0)
        elif want < rows:                     # prune, which is how adaptive width shrinks
            cur = cur[:want]                  # rows are interchangeable at the probe step
            if conf_prev is not None:
                conf_prev = conf_prev[:want]
        rows = want
        used.append(rows)
        cond = condition.expand(rows, -1, -1)

        if ratio <= 0:
            mask_idx = torch.zeros((rows, seq_len), dtype=torch.bool, device=device)
        elif mask_mode == "uncertain" and conf_prev is not None:
            # Mask exactly as many positions as the uniform draw would, but choose them by
            # weighted sampling WITHOUT replacement. The first version scaled a per-position
            # probability, which at a high mask ratio pinned the low-confidence positions to
            # p = 1: they were re-masked every step, never accumulated context and never
            # settled, and WER came out four times worse. Sampling a fixed count keeps the
            # budget identical to flat and still leaves every position a chance to be spared.
            n_mask = int(round(ratio * (seq_len - 1)))
            w = (1.0 - conf_prev).clamp_min(1e-6)
            w = torch.cat([torch.zeros_like(w[:, :1]), w[:, 1:]], dim=1)   # never the BOS slot
            mask_idx = torch.zeros((rows, seq_len), dtype=torch.bool, device=device)
            if n_mask > 0:
                pick = torch.multinomial(w, min(n_mask, seq_len - 1),
                                         replacement=False, generator=gen)
                mask_idx.scatter_(1, pick, True)
        else:
            r = torch.rand((rows, seq_len), device=device, generator=gen)
            mask_idx = r < ratio
            mask_idx[:, 0] = False

        masked = cur.clone()
        masked[mask_idx] = mask_id

        logits = wf.model(idx=masked, condition=cond)

        if step == 0 and first_step_sampling:
            probs = torch.softmax(logits.float() / temperature, dim=-1)
            pred = torch.multinomial(probs.view(-1, probs.size(-1)), 1,
                                     generator=gen).view(rows, seq_len)
        else:
            pred = torch.argmax(logits, dim=-1)

        cur = torch.where(mask_idx, pred, masked)
        if mask_mode == "uncertain" or adaptive is not None:
            conf_prev = torch.softmax(logits.float(), dim=-1).max(dim=-1).values

        if adaptive is not None and step == probe and uncertainty is None:
            a = adaptive
            uncertainty = float((1.0 - conf_prev).mean())
            # base is the TARGET MEAN width, not a ceiling, and u0 is the measured median
            # uncertainty (0.021 over three sets), not a guess. The first version used
            # u0 = 0.15, about seven times too high, so every utterance clipped to k_min.
            k = a.get("base", n_candidates) * (uncertainty / a.get("u0", 0.021)) ** a.get("gamma", 1.0)
            k = int(min(max(round(k), a.get("k_min", 2)), a.get("k_max", n_candidates)))
            scale = k / max(n_candidates, 1)
            for s in range(step + 1, n_steps):       # re-aim the rest of the tree at k
                widths[s] = max(1, min(k, round(widths[s] * scale)))
            widths[n_steps - 1] = k

        if step == 0:
            after_step1 = bool((cur == cur[0:1]).all().item())
        if step == n_steps - 1:
            final_logits = logits

    n_candidates = cur.size(0)

    probs = torch.softmax(final_logits.float(), dim=-1)
    conf = probs.max(dim=-1).values
    logprob = torch.log(
        probs.gather(-1, cur.clamp(max=probs.size(-1) - 1).unsqueeze(-1)).squeeze(-1) + 1e-9
    )
    entropy = -(probs * torch.log(probs + 1e-9)).sum(-1)
    del probs, final_logits

    toks = cur.cpu()
    conf, logprob, entropy = conf.cpu(), logprob.cpu(), entropy.cpu()
    texts = wf.tokenizer.batch_decode(toks, skip_special_tokens=True)
    valid = toks != pad_id

    cands = []
    for i in range(n_candidates):
        v = valid[i]
        if int(v.sum()) == 0:
            v = torch.ones_like(v)
        c = conf[i][v]
        ids = toks[i][v].tolist()
        cands.append(
            Candidate(
                text=texts[i],
                avg_conf=float(c.mean()),
                min_conf=float(c.min()),
                median_conf=float(c.median()),
                mean_logprob=float(logprob[i][v].mean()),
                mean_entropy=float(entropy[i][v].mean()),
                n_tokens=int(v.sum()),
                word_conf=_word_confidences(wf.tokenizer, ids, c.tolist(), texts[i]),
                tokens=ids if save_tokens else None,
            )
        )

    return DecodeResult(
        candidates=cands,
        n_unique=len({c.text for c in cands}),
        identical_after_step1=bool(after_step1),
        n_candidates_used=len(cands),
        uncertainty=uncertainty,
        branch_widths=used,
    )


def candidate_to_dict(c: Candidate) -> dict:
    d = asdict(c)
    for key in ("tokens", "word_conf"):
        if d[key] is None:
            d.pop(key)
    return d


Dependencies and the two upstream repositories, pinned.

In [ ]:
import os, subprocess, sys, time, json, shutil
t_setup = time.time()

def sh(cmd, check=True):
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    if r.returncode != 0:
        print(f"$ {cmd}\n{r.stdout[-3000:]}\n{r.stderr[-3000:]}")
        if check:
            raise RuntimeError(f"failed: {cmd}")
    return r

sh("pip install -q 'lightning>=2.1' lightning-utilities rapidfuzz jiwer einops sentencepiece "
   "omegaconf whisper-normalizer")
UP = "/tmp/upstream"
os.makedirs(UP, exist_ok=True)
PINS = {"Whisfusion": ("https://github.com/taeyoun811/Whisfusion.git", "aa9afe3688ccd15ebf096ec9845d67925d1a3aea"),
        "drax": ("https://github.com/aiola-lab/drax.git", "ffab757b1c88d9c6cdf9912d543032f98fc085e3")}
upstream = {}
for name, (url, pin) in PINS.items():
    d = f"{UP}/{name}"
    if not os.path.exists(d):
        sh(f"git clone -q {url} {d}")
        sh(f"git -C {d} checkout -q {pin}", check=False)
    upstream[name] = sh(f"git -C {d} rev-parse HEAD").stdout.strip()
print("upstream:", upstream)

import importlib.util
assert importlib.util.find_spec("flash_attn") is None, "flash-attn must not be installed on T4 (see wf_compat)"
import torch
print("torch", torch.__version__, "gpus", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
print(f"setup {time.time() - t_setup:.0f} s")


Identity check, then a sanity pass over every tree arm. Needs lit_gpt, so it has to come after the setup above. **If it fails, stop -- do not run the ablation.**

In [ ]:
# The campaign runs as a subprocess with its own PYTHONPATH; this cell runs in the
# kernel, so it has to reproduce that environment before importing anything.
import os, sys
CODE = "/kaggle/working/code/src"
WF = f"{UP}/Whisfusion/src"
for pth in (WF, CODE):
    if pth not in sys.path:
        sys.path.insert(0, pth)
os.environ.setdefault("HF_HOME", "/tmp/hf")          # same cache the campaign will use
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
assert os.path.isdir(WF), f"Whisfusion checkout missing at {WF}; did the setup cell run?"

import check_decode
rc = check_decode.main()
assert rc != 1, "flat sampling changed -- do not run the ablation"


The campaign. Progress is printed every 10 minutes; worker logs are in `campaign/logs/`.

In [ ]:
CODE = "/kaggle/working/code/src"
ROOT = "/kaggle/working/campaign"
DATA = "/tmp/campaign_data"
cfg = dict(CONFIG, mode=MODE, run_hours=RUN_HOURS - (time.time() - NB_START) / 3600, upstream=upstream)
cfg_path = "/kaggle/working/campaign_config.json"
with open(cfg_path, "w") as f:
    json.dump(cfg, f, indent=2)
print(f"campaign budget {cfg['run_hours']:.2f} h")

env = dict(os.environ, PYTHONPATH=f"{CODE}:{UP}/Whisfusion/src", HF_HOME="/tmp/hf",
           DRAX_ROOT=f"{UP}/drax", TOKENIZERS_PARALLELISM="false", HF_HUB_DISABLE_PROGRESS_BARS="1",
           TRANSFORMERS_VERBOSITY="error", PYTHONUNBUFFERED="1")
p = subprocess.Popen([sys.executable, "-m", "campaign.run", "--config", cfg_path, "--root", ROOT,
                      "--data", DATA], env=env, cwd=CODE, stdout=subprocess.PIPE,
                     stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in p.stdout:
    print(line, end="", flush=True)
print("campaign exit code", p.wait())


In [ ]:
from pathlib import Path
rep = Path(ROOT) / "results" / "REPORT.md"
print(rep.read_text() if rep.exists() else "no report")
total = 0
for f in sorted(Path(ROOT).rglob("*")):
    if f.is_file():
        total += f.stat().st_size
print(f"\noutput: {total / 1e9:.2f} GB in {ROOT}")
for f in sorted(Path(ROOT).iterdir()):
    print(f"  {f.name:<32} {f.stat().st_size / 1e6 if f.is_file() else 0:10.1f} MB")
